<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B81%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 크기가 큰 train 쪼개서 넣음.
# 업로드 잘 되었는지 확인
import os

folder = "/content"

for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)

    if os.path.isfile(path):
        size_gb = os.path.getsize(path) / (1024**3)
        print(f"{name:30s} {size_gb:.2f} GB")

sample_submission.csv          0.02 GB
test.parquet                   1.20 GB
train_part_01                  1.46 GB
train_part_02                  1.46 GB
train_part_03                  1.46 GB
train_part_04                  1.46 GB
train_part_05                  1.46 GB
train_part_06                  0.86 GB


In [ ]:
# 데이터 확인
import os
import pyarrow.parquet as pq
import pandas as pd

# =========================
# 1. train 조각 합치기
# =========================

parts = [f"/content/train_part_{i:02d}" for i in range(1, 7)]
train_path = "/content/train.parquet"
test_path = "/content/test.parquet"

with open(train_path, "wb") as outfile:
    for part in parts:
        print("합치는 중:", part)
        with open(part, "rb") as infile:
            while True:
                chunk = infile.read(8 * 1024 * 1024)
                if not chunk:
                    break
                outfile.write(chunk)

print("\ntrain 합치기 완료")
print(f"train 크기: {os.path.getsize(train_path)/(1024**3):.2f} GB")
print(f"test 크기 : {os.path.getsize(test_path)/(1024**3):.2f} GB")


# =========================
# 2. parquet 구조 확인
# =========================

train_pf = pq.ParquetFile(train_path)
test_pf = pq.ParquetFile(test_path)

train_cols = train_pf.schema_arrow.names
test_cols = test_pf.schema_arrow.names

print("\n===== SHAPE =====")
print("TRAIN:", train_pf.metadata.num_rows, "rows x", len(train_cols), "cols")
print("TEST :", test_pf.metadata.num_rows, "rows x", len(test_cols), "cols")

print("\n===== TRAIN에만 있는 컬럼 =====")
target_candidates = [c for c in train_cols if c not in test_cols]
print(target_candidates)

print("\n===== TEST에만 있는 컬럼 =====")
print([c for c in test_cols if c not in train_cols])


# =========================
# 3. 컬럼명 + 자료형
# =========================

print("\n===== TRAIN COLUMNS / TYPES =====")
for field in train_pf.schema_arrow:
    print(f"{field.name:35s} {field.type}")


# =========================
# 4. 일부 데이터만 읽기
# =========================

batch = next(train_pf.iter_batches(batch_size=10000))
sample = batch.to_pandas()

print("\n===== TRAIN HEAD =====")
print(sample.head())

print("\n===== SAMPLE 결측치 비율 TOP 30 =====")
missing = (sample.isna().mean() * 100).sort_values(ascending=False)
print(missing.head(30))

print("\n===== SAMPLE UNIQUE 개수 =====")
unique = sample.nunique(dropna=False).sort_values()
print(unique)


# =========================
# 5. 타깃이 하나라면 전체 분포 확인
# =========================

if len(target_candidates) == 1:
    target = target_candidates[0]

    print("\n===== TARGET =====")
    print("Target column:", target)

    counts = {}

    for batch in train_pf.iter_batches(
        columns=[target],
        batch_size=200000
    ):
        s = batch.to_pandas()[target]
        vc = s.value_counts(dropna=False)

        for k, v in vc.items():
            counts[k] = counts.get(k, 0) + int(v)

    print("\n===== TARGET DISTRIBUTION =====")
    for k, v in sorted(counts.items(), key=lambda x: -x[1]):
        print(k, ":", v)

else:
    print("\n타깃 후보가 1개가 아님:", target_candidates)


# =========================
# 6. 나한테 보내기 좋은 작은 샘플 저장
# =========================

sample.head(5000).to_csv(
    "/content/train_sample_5000.csv",
    index=False
)

test_batch = next(test_pf.iter_batches(batch_size=5000))
test_sample = test_batch.to_pandas()

test_sample.to_csv(
    "/content/test_sample_5000.csv",
    index=False
)

print("\n샘플 저장 완료:")
print("/content/train_sample_5000.csv")
print("/content/test_sample_5000.csv")

합치는 중: /content/train_part_01
합치는 중: /content/train_part_02
합치는 중: /content/train_part_03
합치는 중: /content/train_part_04
합치는 중: /content/train_part_05
합치는 중: /content/train_part_06

train 합치기 완료
train 크기: 8.19 GB
test 크기 : 1.20 GB

===== SHAPE =====
TRAIN: 10704179 rows x 119 cols
TEST : 1527298 rows x 119 cols

===== TRAIN에만 있는 컬럼 =====
['clicked']

===== TEST에만 있는 컬럼 =====
['ID']

===== TRAIN COLUMNS / TYPES =====
gender                              string
age_group                           string
inventory_id                        string
day_of_week                         string
hour                                string
seq                                 string
l_feat_1                            float
l_feat_2                            float
l_feat_3                            float
l_feat_4                            float
l_feat_5                            float
l_feat_6                            float
l_feat_7                            float
l_feat_8                     

In [ ]:
# sample_submission.csv 구조 확인
import pandas as pd

sub = pd.read_csv("/content/sample_submission.csv")

print("shape:", sub.shape)
print(sub.head(10))
print(sub.dtypes)
print("columns:", sub.columns.tolist())

shape: (1527298, 2)
             ID  clicked
0  TEST_0000000        0
1  TEST_0000001        0
2  TEST_0000002        0
3  TEST_0000003        0
4  TEST_0000004        0
5  TEST_0000005        0
6  TEST_0000006        0
7  TEST_0000007        0
8  TEST_0000008        0
9  TEST_0000009        0
ID         object
clicked     int64
dtype: object
columns: ['ID', 'clicked']


# 모델링 파이프라인 작동 확인
- 200만 행 baseline 데이터 생성

- 200만 행으로 빠르게 실험 -> 모델/피처 조합 찾기 -> 1070만 행 전체로 최종 학습 -> test 152만 행 예측

In [ ]:
# 200만 행 baseline 데이터 만들기
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc
import os

train_path = "/content/train.parquet"

pf = pq.ParquetFile(train_path)

# 처음 baseline에서는 seq 제외
cols = [c for c in pf.schema_arrow.names if c != "seq"]

chunks = []
rng = np.random.default_rng(42)

# 전체 데이터의 약 20% → 약 214만 행
for i, batch in enumerate(
    pf.iter_batches(
        columns=cols,
        batch_size=200_000
    )
):
    df = batch.to_pandas()

    # 각 batch에서 랜덤 20%
    idx = rng.random(len(df)) < 0.20
    chunks.append(df.loc[idx])

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches 처리 완료")

sample_train = pd.concat(chunks, ignore_index=True)

del chunks
gc.collect()

print("완료")
print("shape:", sample_train.shape)
print("clicked 분포:")
print(sample_train["clicked"].value_counts())
print("clicked 비율:", sample_train["clicked"].mean())
print(f"메모리: {sample_train.memory_usage(deep=True).sum()/1024**3:.2f} GB")

10 batches 처리 완료
20 batches 처리 완료
30 batches 처리 완료
40 batches 처리 완료
50 batches 처리 완료
완료
shape: (2141187, 118)
clicked 분포:
clicked
0    2100636
1      40551
Name: count, dtype: int64
clicked 비율: 0.018938560714220662
메모리: 1.41 GB


In [ ]:
# categorical 처리 + train/validation 분리
from sklearn.model_selection import train_test_split

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

# LightGBM이 categorical로 인식하도록 변환
for c in cat_cols:
    sample_train[c] = sample_train[c].fillna("MISSING").astype("category")

X = sample_train.drop(columns=["clicked"])
y = sample_train["clicked"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train CTR:", y_train.mean())
print("Valid CTR:", y_valid.mean())


Train: (1712949, 117)
Valid: (428238, 117)
Train CTR: 0.018938684105598007
Valid CTR: 0.018938067149575702


In [ ]:
# LightGBM baseline

!pip -q install lightgbm

import lightgbm as lgb
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective="binary",

    # 불균형 대응
    class_weight="balanced",

    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,

    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",

    categorical_feature=cat_cols,

    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

[LightGBM] [Info] Number of positive: 32441, number of negative: 1680508
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.499152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17687
[LightGBM] [Info] Number of data points in the train set: 1712949, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.73595	valid_0's binary_logloss: 0.581694
[200]	valid_0's auc: 0.736656	valid_0's binary_logloss: 0.56135
Early stopping, best iteration is:
[128]	valid_0's auc: 0.736857	valid_0's binary_logloss: 0.574872


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, n_estimators=2000, n_jobs=-1, num_leaves=63,
               objective='binary', random_state=42, reg_alpha=0.1,
               reg_lambda=1.0, subsample=0.8)

In [ ]:
# validation 성능 계산
from sklearn.metrics import average_precision_score
import numpy as np

pred = model.predict_proba(X_valid)[:, 1]

# 확률 0 또는 1 방지
pred = np.clip(pred, 1e-15, 1 - 1e-15)

# Average Precision
ap = average_precision_score(y_valid, pred)

# Weighted LogLoss
# clicked=0과 clicked=1 각각 50%씩 기여
y_np = y_valid.to_numpy()

pos_loss = -np.mean(np.log(pred[y_np == 1]))
neg_loss = -np.mean(np.log(1 - pred[y_np == 0]))

wll = 0.5 * pos_loss + 0.5 * neg_loss

# 리더보드 점수 형태
score = 0.5 * ap + 0.5 * (1 - wll)

print(f"AP  : {ap:.6f}")
print(f"WLL : {wll:.6f}")
print(f"SCORE: {score:.6f}")

AP  : 0.076063
WLL : 0.601200
SCORE: 0.237431


- 종합점수: 0.2374

(전체 train의 20%만 사용,

가장 정보량이 많을 가능성이 큰 seq 통째로 버리기,

파생변수 거의 없음.

단일 LightGBM 1번 돌리기)

In [ ]:
# seq 파생변수 생성 후, 점수 확인
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

train_path = "/content/train.parquet"

pf = pq.ParquetFile(train_path)

chunks = []
rng = np.random.default_rng(42)

# 이번에는 seq 포함해서 동일하게 약 20% 샘플
for i, batch in enumerate(
    pf.iter_batches(batch_size=200_000)
):
    df = batch.to_pandas()

    mask = rng.random(len(df)) < 0.20
    df = df.loc[mask].copy()

    # ======================
    # seq 파생변수
    # ======================
    seq = df["seq"].fillna("").astype(str)

    # 토큰 개수
    df["seq_len"] = seq.str.count(",") + 1
    df.loc[seq.eq(""), "seq_len"] = 0

    # unique token 개수
    df["seq_nunique"] = seq.apply(
        lambda x: len(set(x.split(","))) if x else 0
    )

    # 반복 정도
    df["seq_unique_ratio"] = (
        df["seq_nunique"] /
        df["seq_len"].replace(0, np.nan)
    ).fillna(0)

    # 원본 seq 문자열은 일단 제거
    df.drop(columns=["seq"], inplace=True)

    chunks.append(df)

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches 완료")

sample_train_seq = pd.concat(chunks, ignore_index=True)

del chunks
gc.collect()

print("shape:", sample_train_seq.shape)
print(sample_train_seq[
    ["seq_len", "seq_nunique", "seq_unique_ratio"]
].describe())

In [ ]:
# sample_train_seq 기준으로 나누기
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import lightgbm as lgb

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

for c in cat_cols:
    sample_train_seq[c] = (
        sample_train_seq[c]
        .fillna("MISSING")
        .astype("category")
    )

X = sample_train_seq.drop(columns=["clicked"])
y = sample_train_seq["clicked"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_seq = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_seq.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    categorical_feature=cat_cols,
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np

pred = model_seq.predict_proba(X_valid)[:, 1]
pred = np.clip(pred, 1e-15, 1 - 1e-15)

y_np = y_valid.to_numpy()

ap = average_precision_score(y_np, pred)

pos_loss = -np.mean(np.log(pred[y_np == 1]))
neg_loss = -np.mean(np.log(1 - pred[y_np == 0]))

wll = 0.5 * pos_loss + 0.5 * neg_loss
score = 0.5 * ap + 0.5 * (1 - wll)

print(f"AP    : {ap:.6f}")
print(f"WLL   : {wll:.6f}")
print(f"SCORE : {score:.6f}")

# 50만 행 baseline 생성

- 50만 행 baseline 데이터 생성

- 50만 행으로 빠르게 실험 -> 모델/피처 조합 찾기 -> 1070만 행 전체로 최종 학습 -> test 152만 행 예측

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

train_path = "/content/train.parquet"

pf = pq.ParquetFile(train_path)

result = []
rng = np.random.default_rng(42)

TARGET_ROWS = 500_000
collected = 0

for i, batch in enumerate(
    pf.iter_batches(batch_size=50_000)
):
    df = batch.to_pandas()

    # 전체 중 약 5%만 랜덤 추출
    mask = rng.random(len(df)) < 0.05
    df = df.loc[mask].copy()

    if len(df) == 0:
        continue

    # seq 문자열
    seq_values = df["seq"].fillna("").astype(str).to_numpy()

    # 토큰 개수 - 중간에 거대한 split 결과를 저장하지 않음
    df["seq_len"] = np.fromiter(
        (0 if s == "" else s.count(",") + 1 for s in seq_values),
        dtype=np.int32,
        count=len(seq_values)
    )

    # unique token 개수
    df["seq_nunique"] = np.fromiter(
        (0 if s == "" else len(set(s.split(","))) for s in seq_values),
        dtype=np.int32,
        count=len(seq_values)
    )

    df["seq_unique_ratio"] = (
        df["seq_nunique"] /
        df["seq_len"].replace(0, np.nan)
    ).fillna(0).astype("float32")

    # 원본 긴 문자열 즉시 제거
    df.drop(columns=["seq"], inplace=True)

    result.append(df)
    collected += len(df)

    del seq_values, df, batch
    gc.collect()

    if (i + 1) % 20 == 0:
        print(f"{i+1} batches / 현재 {collected:,} rows")

    if collected >= TARGET_ROWS:
        break

sample_train_seq = pd.concat(result, ignore_index=True)

# 50만 행까지만
sample_train_seq = sample_train_seq.iloc[:TARGET_ROWS].copy()

del result
gc.collect()

print("\n완료")
print("shape:", sample_train_seq.shape)
print("CTR:", sample_train_seq["clicked"].mean())
print(
    sample_train_seq[
        ["seq_len", "seq_nunique", "seq_unique_ratio"]
    ].describe()
)

20 batches / 현재 49,794 rows
40 batches / 현재 99,627 rows
60 batches / 현재 149,813 rows
80 batches / 현재 200,032 rows
100 batches / 현재 250,046 rows
120 batches / 현재 299,812 rows
140 batches / 현재 349,866 rows
160 batches / 현재 400,134 rows
180 batches / 현재 449,756 rows
200 batches / 현재 499,833 rows

완료
shape: (500000, 121)
CTR: 0.018696
             seq_len    seq_nunique  seq_unique_ratio
count  500000.000000  500000.000000     500000.000000
mean      531.229516      52.774858          0.179050
std       434.526523      23.237082          0.161682
min         1.000000       1.000000          0.003503
25%       182.000000      36.000000          0.085890
50%       440.000000      55.000000          0.125448
75%       779.000000      70.000000          0.205742
max      7422.000000     152.000000          1.000000


In [ ]:
# 50만 행에서, seq 파생변수 X vs seq 파생변수 3개 있음 비교

import numpy as np
import pandas as pd
import lightgbm as lgb

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

# ============================
# 1. categorical 처리
# ============================

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

df = sample_train_seq.copy()

for c in cat_cols:
    df[c] = df[c].fillna("MISSING").astype("category")


# ============================
# 2. 같은 train/valid 행으로 분할
# ============================

train_idx, valid_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=df["clicked"]
)

y_train = df.loc[train_idx, "clicked"]
y_valid = df.loc[valid_idx, "clicked"]


# ============================
# 3. 평가 함수
# ============================

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)
    y_true = np.asarray(y_true)

    ap = average_precision_score(y_true, pred)

    pos_loss = -np.mean(np.log(pred[y_true == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss

    # 지금까지 사용한 비교용 점수
    score = 0.5 * ap + 0.5 * (1 - wll)

    return ap, wll, score


# ============================
# 4. 공통 모델 함수
# ============================

def train_model(X_train, X_valid, name):

    model = LGBMClassifier(
        objective="binary",
        class_weight="balanced",

        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=63,

        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42,
        n_jobs=-1,
        force_col_wise=True
    )

    print(f"\n===== {name} 학습 =====")

    model.fit(
        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",

        categorical_feature=cat_cols,

        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100)
        ]
    )

    pred = model.predict_proba(X_valid)[:, 1]

    ap, wll, score = evaluate(y_valid, pred)

    print(f"\n{name}")
    print(f"AP    : {ap:.6f}")
    print(f"WLL   : {wll:.6f}")
    print(f"SCORE : {score:.6f}")

    return model, pred, ap, wll, score

In [ ]:
# Baseline에서

seq_features = [
    "seq_len",
    "seq_nunique",
    "seq_unique_ratio"
]

base_features = [
    c for c in df.columns
    if c not in ["clicked"] + seq_features
]

X_train_base = df.loc[train_idx, base_features]
X_valid_base = df.loc[valid_idx, base_features]

model_base, pred_base, ap_base, wll_base, score_base = train_model(
    X_train_base,
    X_valid_base,
    "BASELINE"
)


===== BASELINE 학습 =====
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 17691
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.724277	valid_0's binary_logloss: 0.526032
Early stopping, best iteration is:
[66]	valid_0's auc: 0.725385	valid_0's binary_logloss: 0.553065

BASELINE
AP    : 0.063310
WLL   : 0.614311
SCORE : 0.224499


In [ ]:
# seq 포함 모델

seq_model_features = [
    c for c in df.columns
    if c != "clicked"
]

X_train_seq = df.loc[train_idx, seq_model_features]
X_valid_seq = df.loc[valid_idx, seq_model_features]

model_seq, pred_seq, ap_seq, wll_seq, score_seq = train_model(
    X_train_seq,
    X_valid_seq,
    "WITH SEQ FEATURES"
)


===== WITH SEQ FEATURES 학습 =====
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 18334
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 120
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.725619	valid_0's binary_logloss: 0.52434
Early stopping, best iteration is:
[85]	valid_0's auc: 0.725774	valid_0's binary_logloss: 0.535761

WITH SEQ FEATURES
AP    : 0.063428
WLL   : 0.615319
SCORE : 0.224055


In [ ]:
# 비교
print("\n==============================")
print("       최종 비교")
print("==============================")

print(f"Baseline AP    : {ap_base:.6f}")
print(f"Seq AP         : {ap_seq:.6f}")
print(f"변화           : {ap_seq - ap_base:+.6f}")

print()

print(f"Baseline WLL   : {wll_base:.6f}")
print(f"Seq WLL        : {wll_seq:.6f}")
print(f"변화           : {wll_seq - wll_base:+.6f}  <- 낮을수록 좋음")

print()

print(f"Baseline SCORE : {score_base:.6f}")
print(f"Seq SCORE      : {score_seq:.6f}")
print(f"변화           : {score_seq - score_base:+.6f}")


       최종 비교
Baseline AP    : 0.063310
Seq AP         : 0.063428
변화           : +0.000119

Baseline WLL   : 0.614311
Seq WLL        : 0.615319
변화           : +0.001008  <- 낮을수록 좋음

Baseline SCORE : 0.224499
Seq SCORE      : 0.224055
변화           : -0.000445


- 같은 50만 행, 같은 validation에서 AP 변화: 0.063310 -> 0.063428

WLL: 0.614311 -> 0.615319 (안좋아짐)

종합값: 0.224499 -> 0.224055

- seq_len / seq_nunique / seq_unique_ratio 파생변수는 의미 X

In [ ]:
# 50만 행의 feature importance 확인
import pandas as pd

importance = pd.DataFrame({
    "feature": model_base.feature_name_,
    "importance_gain": model_base.booster_.feature_importance(
        importance_type="gain"
    ),
    "importance_split": model_base.booster_.feature_importance(
        importance_type="split"
    )
})

importance = importance.sort_values(
    "importance_gain",
    ascending=False
)

print(importance.head(30).to_string(index=False))

# seq 포함 모델

importance_seq = pd.DataFrame({
    "feature": model_seq.feature_name_,
    "importance_gain": model_seq.booster_.feature_importance(
        importance_type="gain"
    )
})

importance_seq = importance_seq.sort_values(
    "importance_gain",
    ascending=False
)

print(importance_seq.head(40).to_string(index=False))

     feature  importance_gain  importance_split
 history_a_1    260097.626076               259
inventory_id    195470.658348               202
        hour    120000.205170               604
 history_b_2     77704.133896                68
    feat_e_3     28027.779030               124
 history_a_3     26350.308075                70
   age_group     21259.555016                84
    feat_d_4     19155.270996                95
    l_feat_5     18989.652901               103
    feat_b_1     15960.740028                75
history_b_30     15221.000610                62
    feat_c_8     14522.103394                82
   l_feat_15     13912.810989                82
 history_a_2     13304.591957                32
    l_feat_2     12875.987022                40
    feat_c_2     12544.751602                78
    l_feat_6     12476.254013                72
   l_feat_10     11856.975075                68
    feat_b_5     11426.317017                58
   feat_a_14     10839.485947           

In [ ]:
print(
    importance_seq[
        importance_seq["feature"].isin(
            ["seq_len", "seq_nunique", "seq_unique_ratio"]
        )
    ]
)

              feature  importance_gain
118       seq_nunique      8343.669685
119  seq_unique_ratio      7406.495964
117           seq_len      5992.601875


In [ ]:
# feature engineering
# 원래 데이터에서 모델이 더 쉽게 패턴 찾을 수 있도록 새로운 변수 생성 작업

import numpy as np
import pandas as pd

# 다시 깨끗하게 시작
df_fe = df.copy()

# =========================
# 1. hour 주기형 encoding
# =========================
hour_num = pd.to_numeric(
    df_fe["hour"].astype("string"),
    errors="coerce"
).fillna(0)

df_fe["hour_sin"] = np.sin(2 * np.pi * hour_num / 24).astype("float32")
df_fe["hour_cos"] = np.cos(2 * np.pi * hour_num / 24).astype("float32")


# =========================
# 2. feat_e_3 결측 여부
# =========================
df_fe["feat_e_3_missing"] = (
    df_fe["feat_e_3"].isna().astype("int8")
)


# =========================
# 3. categorical count / frequency encoding
# =========================
freq_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

for c in freq_cols:

    # categorical 문제 피하려고 string으로 변환해서 계산
    train_values = (
        df_fe.loc[train_idx, c]
        .astype("string")
        .fillna("MISSING")
    )

    valid_values = (
        df_fe.loc[valid_idx, c]
        .astype("string")
        .fillna("MISSING")
    )

    # train fold에서만 기준 생성
    counts = train_values.value_counts()
    freqs = counts / len(train_values)

    # COUNT
    df_fe.loc[train_idx, f"{c}_count"] = (
        train_values
        .map(counts)
        .astype("float32")
        .to_numpy()
    )

    df_fe.loc[valid_idx, f"{c}_count"] = (
        valid_values
        .map(counts)
        .astype("float32")
        .fillna(0)
        .to_numpy()
    )

    # FREQUENCY
    df_fe.loc[train_idx, f"{c}_freq"] = (
        train_values
        .map(freqs)
        .astype("float32")
        .to_numpy()
    )

    df_fe.loc[valid_idx, f"{c}_freq"] = (
        valid_values
        .map(freqs)
        .astype("float32")
        .fillna(0)
        .to_numpy()
    )

print("완료 ✅")

new_features = [
    c for c in df_fe.columns
    if c not in df.columns
]

print(new_features)
print("추가 피처 개수:", len(new_features))

완료 ✅
['hour_sin', 'hour_cos', 'feat_e_3_missing', 'gender_count', 'gender_freq', 'age_group_count', 'age_group_freq', 'inventory_id_count', 'inventory_id_freq', 'day_of_week_count', 'day_of_week_freq', 'hour_count', 'hour_freq']
추가 피처 개수: 13


In [ ]:
# LightGBM 모델 설정, 학습
features_fe = [
    c for c in df_fe.columns
    if c not in [
        "clicked",
        "seq_len",
        "seq_nunique",
        "seq_unique_ratio"
    ]
]

X_train_fe = df_fe.loc[train_idx, features_fe]
X_valid_fe = df_fe.loc[valid_idx, features_fe]

model_fe, pred_fe, ap_fe, wll_fe, score_fe = train_model(
    X_train_fe,
    X_valid_fe,
    "COUNT + FREQ + HOUR"
)


===== COUNT + FREQ + HOUR 학습 =====
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 17854
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 130
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.723404	valid_0's binary_logloss: 0.524901
Early stopping, best iteration is:
[88]	valid_0's auc: 0.723778	valid_0's binary_logloss: 0.533704

COUNT + FREQ + HOUR
AP    : 0.064147
WLL   : 0.616762
SCORE : 0.223692


- Baseline: 0.224499

- COUNT+FREQ+HOUR: 0.223692

- Seq 3개: 0.224055

- 원본 117개 피처 baseline이 제일 좋음.

In [ ]:
# baseline 모델 300번 학습, 몇 번째 tree에서 최고 점수 확인
# 50만행에서 시행
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

# seq 파생변수 없이 원본 baseline feature
base_features = [
    c for c in df.columns
    if c not in [
        "clicked",
        "seq_len",
        "seq_nunique",
        "seq_unique_ratio"
    ]
]

X_train_base = df.loc[train_idx, base_features]
X_valid_base = df.loc[valid_idx, base_features]

# 300 trees까지 무조건 학습
model_scan = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_scan.fit(
    X_train_base,
    y_train,
    categorical_feature=cat_cols
)

print("학습 완료")

[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 17691
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
학습 완료


In [ ]:
# 앞에서 300개 트리까지 학습한 model_scan으로
#몇 번째 트리까지 사용했을 때의 점수가 가장 좋은지 확인
results = []

y_true = y_valid.to_numpy()

for n_iter in range(20, 301, 10):

    pred = model_scan.predict_proba(
        X_valid_base,
        num_iteration=n_iter
    )[:, 1]

    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    # AP
    ap = average_precision_score(y_true, pred)

    # Weighted LogLoss
    pos_loss = -np.mean(np.log(pred[y_true == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss

    # 대회 비교용 score
    score = 0.5 * ap + 0.5 * (1 - wll)

    results.append({
        "iteration": n_iter,
        "AP": ap,
        "WLL": wll,
        "SCORE": score
    })

results_df = pd.DataFrame(results)

print("===== SCORE TOP 10 =====")
print(
    results_df
    .sort_values("SCORE", ascending=False)
    .head(10)
    .to_string(index=False)
)

===== SCORE TOP 10 =====
 iteration       AP      WLL    SCORE
        70 0.063717 0.614716 0.224500
        80 0.063572 0.615242 0.224165
        90 0.064017 0.616023 0.223997
        60 0.062580 0.614951 0.223815
       100 0.064020 0.617226 0.223397
        50 0.061955 0.615463 0.223246
       110 0.064120 0.618875 0.222622
       120 0.064578 0.619491 0.222544
        40 0.062253 0.617376 0.222438
       130 0.063894 0.620686 0.221604


- 최적 iteration: 70쯤

- 기존 baseline early stopping 66 -> 0.224499

- iteration쪽 확인 필요 X

In [ ]:
# 강한 범주형 변수끼리의 interaction 생성
df_inter = df.copy()

interaction_pairs = [
    ("inventory_id", "hour"),
    ("inventory_id", "age_group"),
    ("inventory_id", "day_of_week"),
    ("age_group", "hour"),
    ("day_of_week", "hour")
]

interaction_cols = []

for a, b in interaction_pairs:
    new_col = f"{a}_X_{b}"

    df_inter[new_col] = (
        df_inter[a].astype("string").fillna("MISSING")
        + "_"
        + df_inter[b].astype("string").fillna("MISSING")
    ).astype("category")

    interaction_cols.append(new_col)

print(interaction_cols)

['inventory_id_X_hour', 'inventory_id_X_age_group', 'inventory_id_X_day_of_week', 'age_group_X_hour', 'day_of_week_X_hour']


In [ ]:
# 원래 categorical 5개 + 새 interaction 5개 -> categorical
cat_cols_inter = cat_cols + interaction_cols

features_inter = [
    c for c in df_inter.columns
    if c not in [
        "clicked",
        "seq_len",
        "seq_nunique",
        "seq_unique_ratio"
    ]
]

X_train_inter = df_inter.loc[train_idx, features_inter]
X_valid_inter = df_inter.loc[valid_idx, features_inter]

In [ ]:
# train_model() 함수가 기존 cat_cols만 사용하도록 설정되어있음.
# 새 모델 돌리기
from lightgbm import LGBMClassifier
import lightgbm as lgb
import numpy as np
from sklearn.metrics import average_precision_score

model_inter = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_inter.fit(
    X_train_inter,
    y_train,

    categorical_feature=cat_cols_inter
)

print("학습 완료")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 18608
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 122
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
학습 완료


In [ ]:
# 70 iteration 기준으로 비교
pred_inter = model_inter.predict_proba(
    X_valid_inter,
    num_iteration=70
)[:, 1]

pred_inter = np.clip(pred_inter, 1e-15, 1 - 1e-15)

y_true = y_valid.to_numpy()

ap_inter = average_precision_score(y_true, pred_inter)

pos_loss = -np.mean(np.log(pred_inter[y_true == 1]))
neg_loss = -np.mean(np.log(1 - pred_inter[y_true == 0]))

wll_inter = 0.5 * pos_loss + 0.5 * neg_loss
score_inter = 0.5 * ap_inter + 0.5 * (1 - wll_inter)

print(f"AP    : {ap_inter:.6f}")
print(f"WLL   : {wll_inter:.6f}")
print(f"SCORE : {score_inter:.6f}")

print("\nBaseline SCORE : 0.224500")
print(f"Interaction    : {score_inter:.6f}")
print(f"변화           : {score_inter - 0.224500:+.6f}")

AP    : 0.048321
WLL   : 0.688358
SCORE : 0.179982

Baseline SCORE : 0.224500
Interaction    : 0.179982
변화           : -0.044518


In [ ]:
results_inter = []

y_true = y_valid.to_numpy()

for n_iter in range(20, 301, 10):

    pred = model_inter.predict_proba(
        X_valid_inter,
        num_iteration=n_iter
    )[:, 1]

    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos_loss = -np.mean(np.log(pred[y_true == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss
    score = 0.5 * ap + 0.5 * (1 - wll)

    results_inter.append({
        "iteration": n_iter,
        "AP": ap,
        "WLL": wll,
        "SCORE": score
    })

results_inter_df = pd.DataFrame(results_inter)

print(
    results_inter_df
    .sort_values("SCORE", ascending=False)
    .head(10)
    .to_string(index=False)
)

 iteration       AP      WLL    SCORE
        20 0.045933 0.650148 0.197892
        30 0.047555 0.652142 0.197707
        40 0.048196 0.659512 0.194342
        50 0.048142 0.669263 0.189439
        60 0.048695 0.676960 0.185867
        70 0.048321 0.688358 0.179982
        80 0.047514 0.700834 0.173340
        90 0.046810 0.713440 0.166685
       100 0.046206 0.726488 0.159859
       110 0.045801 0.740303 0.152749


- inventory_id x hour같은 조합을 문자열 categorical로 직접 생성 -> LightGBM 입장에서 범주가 훨씬 잘게 쪼개짐. 현재 40만 training row에서 해당 데이터가 적어져서 일반화 X

- 이 과정 버리기

In [ ]:
# 기존 50만 행 데이터
# 값이 비어 있는지 여부를 새 피처 3개로 추가

df_null = df.copy()

df_null["null_gender"] = df_null["gender"].isna().astype("int8")
df_null["null_feat_a"] = df_null["feat_a_1"].isna().astype("int8")
df_null["null_feat_e"] = df_null["feat_e_3"].isna().astype("int8")

null_features = [
    c for c in df_null.columns
    if c not in [
        "clicked",
        "seq_len",
        "seq_nunique",
        "seq_unique_ratio"
    ]
]

X_train_null = df_null.loc[train_idx, null_features]
X_valid_null = df_null.loc[valid_idx, null_features]

print("추가된 피처:", ["null_gender", "null_feat_a", "null_feat_e"])
print(X_train_null.shape)

추가된 피처: ['null_gender', 'null_feat_a', 'null_feat_e']
(400000, 120)


In [ ]:
# null 피처 3개 넣은 LightGBM 학습
# baseline과의 비교

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
import numpy as np

model_null = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    n_estimators=70,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_null.fit(
    X_train_null,
    y_train,
    categorical_feature=cat_cols
)

print("학습 완료")

[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 17695
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
학습 완료


In [ ]:
# 평가
pred_null = model_null.predict_proba(X_valid_null)[:, 1]
pred_null = np.clip(pred_null, 1e-15, 1 - 1e-15)

y_true = y_valid.to_numpy()

# AP
ap_null = average_precision_score(y_true, pred_null)

# Weighted LogLoss
pos_loss = -np.mean(np.log(pred_null[y_true == 1]))
neg_loss = -np.mean(np.log(1 - pred_null[y_true == 0]))
wll_null = 0.5 * pos_loss + 0.5 * neg_loss

print("===== NULL FEATURE MODEL =====")
print(f"AP  : {ap_null:.6f}")
print(f"WLL : {wll_null:.6f}")

print("\n===== BASELINE (70 iter) =====")
print("AP  : 0.063717")
print("WLL : 0.614716")

===== NULL FEATURE MODEL =====
AP  : 0.063379
WLL : 0.613800

===== BASELINE (70 iter) =====
AP  : 0.063717
WLL : 0.614716


- AP: 0.063717 -> 0.063379 (조금 나빠짐)

- WLL: 0.6114716 -> 0.613000 (조금 좋아짐)

In [ ]:
# baseline 예측값, null 모델 예측값 섞기
import numpy as np
from sklearn.metrics import average_precision_score

# baseline / null 모델 예측값
# alpha = baseline 비중
# 1-alpha = null 모델 비중

results_blend = []

for alpha in np.arange(0, 1.01, 0.05):

    pred_blend = (
        alpha * pred_base
        + (1 - alpha) * pred_null
    )

    pred_blend = np.clip(pred_blend, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred_blend)

    pos_loss = -np.mean(np.log(pred_blend[y_true == 1]))
    neg_loss = -np.mean(np.log(1 - pred_blend[y_true == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    results_blend.append((alpha, ap, wll))


print("alpha = baseline 비중\n")

for alpha, ap, wll in results_blend:
    print(
        f"alpha={alpha:.2f} | "
        f"AP={ap:.6f} | "
        f"WLL={wll:.6f}"
    )

alpha = baseline 비중

alpha=0.00 | AP=0.063379 | WLL=0.613800
alpha=0.05 | AP=0.063416 | WLL=0.613655
alpha=0.10 | AP=0.063503 | WLL=0.613527
alpha=0.15 | AP=0.063561 | WLL=0.613418
alpha=0.20 | AP=0.063589 | WLL=0.613327
alpha=0.25 | AP=0.063679 | WLL=0.613254
alpha=0.30 | AP=0.063709 | WLL=0.613200
alpha=0.35 | AP=0.063665 | WLL=0.613163
alpha=0.40 | AP=0.063689 | WLL=0.613144
alpha=0.45 | AP=0.063675 | WLL=0.613142
alpha=0.50 | AP=0.063871 | WLL=0.613159
alpha=0.55 | AP=0.063829 | WLL=0.613194
alpha=0.60 | AP=0.063797 | WLL=0.613246
alpha=0.65 | AP=0.063756 | WLL=0.613316
alpha=0.70 | AP=0.063730 | WLL=0.613405
alpha=0.75 | AP=0.063710 | WLL=0.613511
alpha=0.80 | AP=0.063664 | WLL=0.613635
alpha=0.85 | AP=0.063563 | WLL=0.613776
alpha=0.90 | AP=0.063489 | WLL=0.613936
alpha=0.95 | AP=0.063417 | WLL=0.614114
alpha=1.00 | AP=0.063310 | WLL=0.614311


- baseline 50% + null 모델 50% 앙상블

- alpha=0.50에서, AP=0.063871, WLL=0.613159

In [ ]:
# 70회 baseline + 70회 null 모델
# baseline 모델의 70 iteration 예측
pred_base70 = model_scan.predict_proba(
    X_valid_base,
    num_iteration=70
)[:, 1]

pred_base70 = np.clip(pred_base70, 1e-15, 1 - 1e-15)

results_blend70 = []

for alpha in np.arange(0, 1.01, 0.05):

    pred_blend = (
        alpha * pred_base70
        + (1 - alpha) * pred_null
    )

    pred_blend = np.clip(pred_blend, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred_blend)

    pos_loss = -np.mean(np.log(pred_blend[y_true == 1]))
    neg_loss = -np.mean(np.log(1 - pred_blend[y_true == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    results_blend70.append((alpha, ap, wll))

print("alpha = baseline70 비중\n")

for alpha, ap, wll in results_blend70:
    print(
        f"alpha={alpha:.2f} | "
        f"AP={ap:.6f} | "
        f"WLL={wll:.6f}"
    )


alpha = baseline70 비중

alpha=0.00 | AP=0.063379 | WLL=0.613800
alpha=0.05 | AP=0.063423 | WLL=0.613677
alpha=0.10 | AP=0.063560 | WLL=0.613572
alpha=0.15 | AP=0.063625 | WLL=0.613484
alpha=0.20 | AP=0.063662 | WLL=0.613414
alpha=0.25 | AP=0.063728 | WLL=0.613362
alpha=0.30 | AP=0.063709 | WLL=0.613328
alpha=0.35 | AP=0.063735 | WLL=0.613311
alpha=0.40 | AP=0.063827 | WLL=0.613312
alpha=0.45 | AP=0.063833 | WLL=0.613331
alpha=0.50 | AP=0.063841 | WLL=0.613368
alpha=0.55 | AP=0.063841 | WLL=0.613422
alpha=0.60 | AP=0.063805 | WLL=0.613494
alpha=0.65 | AP=0.064038 | WLL=0.613584
alpha=0.70 | AP=0.064025 | WLL=0.613691
alpha=0.75 | AP=0.063987 | WLL=0.613817
alpha=0.80 | AP=0.063938 | WLL=0.613960
alpha=0.85 | AP=0.063883 | WLL=0.614122
alpha=0.90 | AP=0.063816 | WLL=0.614302
alpha=0.95 | AP=0.063764 | WLL=0.614500
alpha=1.00 | AP=0.063717 | WLL=0.614716


- AP 최고: alpha--=0.65 -> 0.64038

- WLL 최저: alpha=0.35 -> 0.613311

- baseline 45% + null model 55% (alpha=0.45) 임시 앙상블로 저장

- 최종 제출 시, alpha는 validation 여러 개 값에서 확인

In [ ]:
# seq의 위치 토큰 파생변수
# 10만 행으로 seq에서 실제 내용 정보 뽑기 (50만행 RAM 터짐)
# seq의 특정 위치에 어떤 토큰이 있는지 확인

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

pf = pq.ParquetFile("/content/train.parquet")

rng = np.random.default_rng(42)
chunks = []

TARGET_ROWS = 100_000
collected = 0

for batch in pf.iter_batches(batch_size=20_000):

    tmp = batch.to_pandas()

    # 전체 데이터에서 약 1% 샘플
    mask = rng.random(len(tmp)) < 0.01
    tmp = tmp.loc[mask].copy()

    if len(tmp) == 0:
        del batch, tmp
        continue

    seqs = tmp["seq"].fillna("").astype(str).to_numpy()

    # 메모리에 큰 split 결과를 저장하지 않고 한 행씩 작은 값만 추출
    first_token = []
    second_token = []
    last_token = []
    seq_len = []
    seq_nunique = []

    for s in seqs:
        if s == "":
            first_token.append("MISSING")
            second_token.append("MISSING")
            last_token.append("MISSING")
            seq_len.append(0)
            seq_nunique.append(0)
        else:
            tokens = s.split(",")

            first_token.append(tokens[0])
            second_token.append(tokens[1] if len(tokens) > 1 else "MISSING")
            last_token.append(tokens[-1])

            seq_len.append(len(tokens))
            seq_nunique.append(len(set(tokens)))

    tmp["seq_first"] = first_token
    tmp["seq_second"] = second_token
    tmp["seq_last"] = last_token
    tmp["seq_len"] = np.asarray(seq_len, dtype=np.int32)
    tmp["seq_nunique"] = np.asarray(seq_nunique, dtype=np.int32)

    # 가장 중요한 부분: 긴 문자열 즉시 삭제
    tmp.drop(columns=["seq"], inplace=True)

    chunks.append(tmp)
    collected += len(tmp)

    del seqs, first_token, second_token, last_token
    del seq_len, seq_nunique, batch, tmp
    gc.collect()

    if collected >= TARGET_ROWS:
        break

seq_sample = pd.concat(chunks, ignore_index=True).iloc[:TARGET_ROWS].copy()

del chunks
gc.collect()

print("완료")
print("shape:", seq_sample.shape)
print("CTR:", seq_sample["clicked"].mean())

print(seq_sample[
    ["seq_first", "seq_second", "seq_last", "seq_len", "seq_nunique"]
].head())

완료 ✅
shape: (100000, 123)
CTR: 0.01843
  seq_first seq_second seq_last  seq_len  seq_nunique
0         9        138       35      539           44
1       144         57       35      585           65
2        57        463      479      114           20
3       321        269      479      461           65
4         9        269      479     1071           72


In [ ]:
# 앞에서 생성한 seq 정보 LightGBM으로 확인
# 원래 피처 vs 원래 피처 + seq_first, seq_second, seq_last, seq_len, seq_nunique 사용
import pandas as pd
import numpy as np
import lightgbm as lgb

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

# 복사본
df100 = seq_sample.copy()

base_cat = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

seq_cat = [
    "seq_first",
    "seq_second",
    "seq_last"
]

# LightGBM categorical 타입으로 변환
for c in base_cat + seq_cat:
    df100[c] = (
        df100[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

# 똑같은 행으로 train / validation 나누기
train_idx100, valid_idx100 = train_test_split(
    np.arange(len(df100)),
    test_size=0.20,
    random_state=42,
    stratify=df100["clicked"]
)

y_train100 = df100.loc[train_idx100, "clicked"]
y_valid100 = df100.loc[valid_idx100, "clicked"]

print("Train:", len(train_idx100))
print("Valid:", len(valid_idx100))
print("Train CTR:", y_train100.mean())
print("Valid CTR:", y_valid100.mean())

Train: 80000
Valid: 20000
Train CTR: 0.018425
Valid CTR: 0.01845


In [ ]:
# Baseline 모델
seq_features = [
    "seq_first",
    "seq_second",
    "seq_last",
    "seq_len",
    "seq_nunique"
]

base_features100 = [
    c for c in df100.columns
    if c not in ["clicked"] + seq_features
]

X_train_base100 = df100.loc[train_idx100, base_features100]
X_valid_base100 = df100.loc[valid_idx100, base_features100]

model_base100 = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    n_estimators=70,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_base100.fit(
    X_train_base100,
    y_train100,
    categorical_feature=base_cat
)

print("Baseline 학습 완료")

[LightGBM] [Info] Number of positive: 1474, number of negative: 78526
[LightGBM] [Info] Total Bins 17587
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Baseline 학습 완료


In [ ]:
# Seq 정보까지 넣은 모델
all_features100 = [
    c for c in df100.columns
    if c != "clicked"
]

X_train_seq100 = df100.loc[train_idx100, all_features100]
X_valid_seq100 = df100.loc[valid_idx100, all_features100]

model_seq100 = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    n_estimators=70,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model_seq100.fit(
    X_train_seq100,
    y_train100,
    categorical_feature=base_cat + seq_cat
)

print("SEQ 모델 학습 완료")

[LightGBM] [Info] Number of positive: 1474, number of negative: 78526
[LightGBM] [Info] Total Bins 18251
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 122
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
SEQ 모델 학습 완료


In [ ]:
# 두 모델 비교
# 10만 행에서 baseline vs seq-position 비교
def evaluate_model(model, X, y):
    pred = model.predict_proba(X)[:, 1]
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    y_np = y.to_numpy()

    ap = average_precision_score(y_np, pred)

    pos_loss = -np.mean(np.log(pred[y_np == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y_np == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    return ap, wll

ap_base100, wll_base100 = evaluate_model(
    model_base100,
    X_valid_base100,
    y_valid100
)

ap_seq100, wll_seq100 = evaluate_model(
    model_seq100,
    X_valid_seq100,
    y_valid100
)

print("===== BASELINE =====")
print(f"AP  : {ap_base100:.6f}")
print(f"WLL : {wll_base100:.6f}")

print("\n===== SEQ POSITION FEATURES =====")
print(f"AP  : {ap_seq100:.6f}")
print(f"WLL : {wll_seq100:.6f}")

print("\n===== 변화 =====")
print(f"AP 변화  : {ap_seq100 - ap_base100:+.6f}")
print(f"WLL 변화 : {wll_seq100 - wll_base100:+.6f}  (음수면 개선)")

===== BASELINE =====
AP  : 0.058892
WLL : 0.677499

===== SEQ POSITION FEATURES =====
AP  : 0.070592
WLL : 0.702299

===== 변화 =====
AP 변화  : +0.011701
WLL 변화 : +0.024801  (음수면 개선)


- AP: 0.058892 → 0.070592 (많이 좋아짐)

- WLL: 0.677499 → 0.702299 (나빠짐)

In [ ]:
# 앞 셀: 100,000행 모델에서 70 trees 고정
#-> 50만 행에서 최적값이 아닐 가능성
# model_base100, model_seq100에서 10~70 iteration 확인

import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def get_ap_wll(y, pred):
    y = np.asarray(y)
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y, pred)

    pos_loss = -np.mean(np.log(pred[y == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    return ap, wll


rows = []
y100 = y_valid100.to_numpy()

for n in range(10, 71, 5):

    # baseline
    p_base = model_base100.predict_proba(
        X_valid_base100,
        num_iteration=n
    )[:, 1]

    ap_b, wll_b = get_ap_wll(y100, p_base)

    # seq model
    p_seq = model_seq100.predict_proba(
        X_valid_seq100,
        num_iteration=n
    )[:, 1]

    ap_s, wll_s = get_ap_wll(y100, p_seq)

    rows.append({
        "iter": n,
        "base_AP": ap_b,
        "base_WLL": wll_b,
        "seq_AP": ap_s,
        "seq_WLL": wll_s
    })

scan = pd.DataFrame(rows)

print(scan.to_string(index=False))

 iter  base_AP  base_WLL   seq_AP  seq_WLL
   10 0.044655  0.658715 0.056697 0.659334
   15 0.051393  0.651124 0.058159 0.653766
   20 0.056623  0.647708 0.058995 0.650358
   25 0.056017  0.646375 0.061493 0.648919
   30 0.057343  0.644998 0.061398 0.650580
   35 0.060995  0.645958 0.061479 0.655116
   40 0.060396  0.647407 0.061120 0.657803
   45 0.061432  0.650854 0.063658 0.662841
   50 0.059717  0.654792 0.064836 0.669185
   55 0.058877  0.659829 0.067800 0.677476
   60 0.058022  0.666982 0.073243 0.687342
   65 0.057960  0.672814 0.070152 0.695684
   70 0.058892  0.677499 0.070592 0.702299


- 60 iteration에서 AP=0.073243까지 증가 (가장 좋은 값)

- 50~70으로 갈수록 WLL 급격히 나빠짐 (과격하게 예측하는 과적합)

In [ ]:
# 60 iteration seq 모델의 순위 성능 그대로 유지 + 확률 보정 필요
# 확률값 보정하여 WLL 많이 낮추기

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# AP가 가장 좋았던 seq 모델 60 iteration 예측
p = model_seq100.predict_proba(
    X_valid_seq100,
    num_iteration=60
)[:, 1]

p = np.clip(p, 1e-8, 1 - 1e-8)
y = y_valid100.to_numpy()

# logit
logit = np.log(p / (1 - p))

results_temp = []

# T > 1이면 예측확률을 0.5 방향으로 완화
for T in np.arange(1.0, 5.01, 0.1):

    calibrated = 1 / (1 + np.exp(-logit / T))

    ap = average_precision_score(y, calibrated)

    pos_loss = -np.mean(np.log(calibrated[y == 1]))
    neg_loss = -np.mean(np.log(1 - calibrated[y == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss

    results_temp.append({
        "T": T,
        "AP": ap,
        "WLL": wll
    })

temp_df = pd.DataFrame(results_temp)

print("===== WLL BEST 10 =====")
print(
    temp_df
    .sort_values("WLL")
    .head(10)
    .to_string(index=False)
)

===== WLL BEST 10 =====
  T       AP      WLL
2.0 0.073243 0.656366
2.1 0.073243 0.656419
1.9 0.073243 0.656489
2.2 0.073243 0.656605
1.8 0.073243 0.656843
2.3 0.073243 0.656889
2.4 0.073243 0.657248
1.7 0.073243 0.657507
2.5 0.073243 0.657661
2.6 0.073243 0.658114


- seq 60 iteration 시 , AP=0.073243, WLL=0.687342

- T=2.0으로 보정 시, AP=0.073243, WLL=0.656366

In [ ]:
# seq_first / seq_second / seq_last 효과를 50만 행에서 한 번 더 검증
# 같은 효과인지 확인
# 50만 행 실험용 데이터 만드는 전처리 작업

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

pf = pq.ParquetFile("/content/train.parquet")

rng = np.random.default_rng(42)
chunks = []

TARGET_ROWS = 500_000
collected = 0

for i, batch in enumerate(pf.iter_batches(batch_size=20_000)):

    tmp = batch.to_pandas()

    # 전체에서 약 5% 추출
    mask = rng.random(len(tmp)) < 0.05
    tmp = tmp.loc[mask].copy()

    if len(tmp) == 0:
        del batch, tmp
        continue

    seqs = tmp["seq"].fillna("").astype(str).to_numpy()

    first = []
    second = []
    last = []

    for s in seqs:
        if not s:
            first.append("MISSING")
            second.append("MISSING")
            last.append("MISSING")
        else:
            tokens = s.split(",")

            first.append(tokens[0])
            second.append(tokens[1] if len(tokens) > 1 else "MISSING")
            last.append(tokens[-1])

    tmp["seq_first"] = first
    tmp["seq_second"] = second
    tmp["seq_last"] = last

    # 긴 원본은 바로 제거
    tmp.drop(columns=["seq"], inplace=True)

    chunks.append(tmp)
    collected += len(tmp)

    del seqs, first, second, last, tmp, batch
    gc.collect()

    if (i + 1) % 50 == 0:
        print(f"{i+1} batches / {collected:,} rows")

    if collected >= TARGET_ROWS:
        break

df500 = pd.concat(chunks, ignore_index=True).iloc[:TARGET_ROWS].copy()

del chunks
gc.collect()

print("완료")
print("shape:", df500.shape)
print("CTR:", df500["clicked"].mean())

50 batches / 49,794 rows
100 batches / 99,627 rows
150 batches / 149,813 rows
200 batches / 200,032 rows
250 batches / 250,046 rows
300 batches / 299,812 rows
350 batches / 349,866 rows
400 batches / 400,134 rows
450 batches / 449,756 rows
500 batches / 499,833 rows
완료
shape: (500000, 121)
CTR: 0.018696


In [ ]:
# 같은 50만 행 train/validation으로 나누기
# categorical 처리
# baseline, seq 모델 120 tree까지 학습
# iteration 미리 고정 X

import numpy as np
from sklearn.model_selection import train_test_split

base_cat = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

seq_cat = [
    "seq_first",
    "seq_second",
    "seq_last"
]

for c in base_cat + seq_cat:
    df500[c] = (
        df500[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

train_idx500, valid_idx500 = train_test_split(
    np.arange(len(df500)),
    test_size=0.20,
    random_state=42,
    stratify=df500["clicked"]
)

y_train500 = df500.loc[train_idx500, "clicked"]
y_valid500 = df500.loc[valid_idx500, "clicked"]

print("Train:", len(train_idx500))
print("Valid:", len(valid_idx500))

Train: 400000
Valid: 100000


In [ ]:
# model_base500, model_seq500 생성 실제 LightGBM 학습 셀 생성

from lightgbm import LGBMClassifier

seq_position = [
    "seq_first",
    "seq_second",
    "seq_last"
]

base_features500 = [
    c for c in df500.columns
    if c not in ["clicked"] + seq_position
]

seq_features500 = [
    c for c in df500.columns
    if c != "clicked"
]

X_train_base500 = df500.loc[train_idx500, base_features500]
X_valid_base500 = df500.loc[valid_idx500, base_features500]

X_train_seq500 = df500.loc[train_idx500, seq_features500]
X_valid_seq500 = df500.loc[valid_idx500, seq_features500]

params = dict(
    objective="binary",
    class_weight="balanced",
    n_estimators=120,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

print("BASELINE 학습 시작")
model_base500 = LGBMClassifier(**params)
model_base500.fit(
    X_train_base500,
    y_train500,
    categorical_feature=base_cat
)

print("SEQ 모델 학습 시작")
model_seq500 = LGBMClassifier(**params)
model_seq500.fit(
    X_train_seq500,
    y_train500,
    categorical_feature=base_cat + seq_cat
)

print("둘 다 학습 완료")

BASELINE 학습 시작
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 17691
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
SEQ 모델 학습 시작
[LightGBM] [Info] Number of positive: 7478, number of negative: 392522
[LightGBM] [Info] Total Bins 18072
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 120
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
둘 다 학습 완료


In [ ]:
# 50만 행 baseline 모델, seq 모델 각 iteration마다 비교
# 최적의 iteration 구하기

from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

def ap_wll(y, pred):
    y = np.asarray(y)
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y, pred)

    pos_loss = -np.mean(np.log(pred[y == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    return ap, wll

y500 = y_valid500.to_numpy()
rows = []

for n in range(20, 121, 10):

    p_base = model_base500.predict_proba(
        X_valid_base500,
        num_iteration=n
    )[:, 1]

    p_seq = model_seq500.predict_proba(
        X_valid_seq500,
        num_iteration=n
    )[:, 1]

    base_ap, base_wll = ap_wll(y500, p_base)
    seq_ap, seq_wll = ap_wll(y500, p_seq)

    rows.append({
        "iter": n,
        "base_AP": base_ap,
        "base_WLL": base_wll,
        "seq_AP": seq_ap,
        "seq_WLL": seq_wll
    })

result500 = pd.DataFrame(rows)
print(result500.to_string(index=False))

 iter  base_AP  base_WLL   seq_AP  seq_WLL
   20 0.059003  0.631074 0.058314 0.632380
   30 0.060098  0.621670 0.059721 0.623289
   40 0.062253  0.617376 0.060441 0.619579
   50 0.061955  0.615463 0.060678 0.618057
   60 0.062580  0.614951 0.060843 0.617533
   70 0.063717  0.614716 0.061670 0.618382
   80 0.063572  0.615242 0.061969 0.619749
   90 0.064017  0.616023 0.061775 0.621548
  100 0.064020  0.617226 0.061854 0.623278
  110 0.064120  0.618875 0.061055 0.625455
  120 0.064578  0.619491 0.060960 0.627499


- 50만 행에서의 결과 eq_first / seq_second / seq_last 버리기

In [ ]:
# 새 모델 학습 없이 baseline 120회 모델 callibration

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# baseline 120 iteration 예측
p = model_base500.predict_proba(
    X_valid_base500,
    num_iteration=120
)[:, 1]

p = np.clip(p, 1e-8, 1 - 1e-8)
y = y_valid500.to_numpy()

logit = np.log(p / (1 - p))

rows = []

for T in np.arange(0.5, 4.01, 0.05):

    calibrated = 1 / (1 + np.exp(-logit / T))

    ap = average_precision_score(y, calibrated)

    pos_loss = -np.mean(np.log(calibrated[y == 1]))
    neg_loss = -np.mean(np.log(1 - calibrated[y == 0]))
    wll = 0.5 * pos_loss + 0.5 * neg_loss

    rows.append({
        "T": T,
        "AP": ap,
        "WLL": wll
    })

cal120 = pd.DataFrame(rows)

print("===== WLL BEST 10 =====")
print(
    cal120.sort_values("WLL")
          .head(10)
          .to_string(index=False)
)

===== WLL BEST 10 =====
   T       AP      WLL
1.15 0.064578 0.618295
1.10 0.064578 0.618378
1.20 0.064578 0.618441
1.05 0.064578 0.618751
1.25 0.064578 0.618769
1.30 0.064578 0.619239
1.00 0.064578 0.619491
1.35 0.064578 0.619823
1.40 0.064578 0.620494
0.95 0.064578 0.620695


In [ ]:
# 확률 보정 2개 파라미터(기울기+이동)
# 70 iteration 안정적인 기준 모델로 유지
# 120 iter은 AP 높은 후보
# validation 절반으로 나눠서 한쪽에서 calibration, 다른 쪽에서 평가

import numpy as np
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid500.to_numpy()

# 70 / 120 iteration 예측
p70 = model_base500.predict_proba(
    X_valid_base500,
    num_iteration=70
)[:, 1]

p120 = model_base500.predict_proba(
    X_valid_base500,
    num_iteration=120
)[:, 1]

p70 = np.clip(p70, 1e-8, 1 - 1e-8)
p120 = np.clip(p120, 1e-8, 1 - 1e-8)

# validation을 calibration용 / 최종 평가용으로 절반 분리
idx = np.arange(len(y))

cal_idx, eval_idx = train_test_split(
    idx,
    test_size=0.5,
    random_state=2026,
    stratify=y
)

# 120 prediction의 logit
z = np.log(p120 / (1 - p120))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

# class-balanced WLL
def calibration_loss(theta):
    a, b = theta

    q = sigmoid(
        a * z[cal_idx] + b
    )

    yy = y[cal_idx]

    pos = -np.mean(np.log(q[yy == 1]))
    neg = -np.mean(np.log(1 - q[yy == 0]))

    return 0.5 * pos + 0.5 * neg

result = minimize(
    calibration_loss,
    x0=[1.0, 0.0],
    method="Nelder-Mead"
)

a, b = result.x

print("calibration a:", a)
print("calibration b:", b)


def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll


# 평가용 절반에서만 비교
y_eval = y[eval_idx]

ap70, wll70 = evaluate(
    y_eval,
    p70[eval_idx]
)

ap120_raw, wll120_raw = evaluate(
    y_eval,
    p120[eval_idx]
)

p120_cal = sigmoid(
    a * z[eval_idx] + b
)

ap120_cal, wll120_cal = evaluate(
    y_eval,
    p120_cal
)

print("\n===== 70 ITER =====")
print(f"AP  : {ap70:.6f}")
print(f"WLL : {wll70:.6f}")

print("\n===== 120 ITER RAW =====")
print(f"AP  : {ap120_raw:.6f}")
print(f"WLL : {wll120_raw:.6f}")

print("\n===== 120 ITER CALIBRATED =====")
print(f"AP  : {ap120_cal:.6f}")
print(f"WLL : {wll120_cal:.6f}")

calibration a: 0.928429154505485
calibration b: 0.23893076715646933

===== 70 ITER =====
AP  : 0.062095
WLL : 0.612474

===== 120 ITER RAW =====
AP  : 0.063979
WLL : 0.616337

===== 120 ITER CALIBRATED =====
AP  : 0.063979
WLL : 0.610212


- 120 iter+calibration이 가장 좋은 결과

In [ ]:
# calibration/evaluation 나누는 seed 여러 개 바꾸어서 결과 그대로인지 확인
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid500.to_numpy()

p70 = model_base500.predict_proba(
    X_valid_base500,
    num_iteration=70
)[:, 1]

p120 = model_base500.predict_proba(
    X_valid_base500,
    num_iteration=120
)[:, 1]

p70 = np.clip(p70, 1e-8, 1 - 1e-8)
p120 = np.clip(p120, 1e-8, 1 - 1e-8)

z120 = np.log(p120 / (1 - p120))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg
    return ap, wll


rows = []

for seed in [1, 42, 100, 2026, 7777]:

    idx = np.arange(len(y))

    cal_idx, eval_idx = train_test_split(
        idx,
        test_size=0.5,
        random_state=seed,
        stratify=y
    )

    def loss(theta):
        a, b = theta
        q = sigmoid(a * z120[cal_idx] + b)

        yy = y[cal_idx]

        pos = -np.mean(np.log(q[yy == 1]))
        neg = -np.mean(np.log(1 - q[yy == 0]))

        return 0.5 * pos + 0.5 * neg

    res = minimize(
        loss,
        x0=[1.0, 0.0],
        method="Nelder-Mead"
    )

    a, b = res.x

    y_eval = y[eval_idx]

    ap70, wll70 = evaluate(
        y_eval,
        p70[eval_idx]
    )

    p120_cal = sigmoid(
        a * z120[eval_idx] + b
    )

    ap120, wll120 = evaluate(
        y_eval,
        p120_cal
    )

    rows.append({
        "seed": seed,
        "a": a,
        "b": b,
        "AP_70": ap70,
        "WLL_70": wll70,
        "AP_120_cal": ap120,
        "WLL_120_cal": wll120,
        "AP_gain": ap120 - ap70,
        "WLL_gain": wll120 - wll70
    })

check = pd.DataFrame(rows)

print(check.to_string(index=False))

print("\n===== 평균 =====")
print(check[
    ["AP_70", "WLL_70", "AP_120_cal", "WLL_120_cal",
     "AP_gain", "WLL_gain"]
].mean())

 seed        a        b    AP_70   WLL_70  AP_120_cal  WLL_120_cal   AP_gain  WLL_gain
    1 0.904388 0.242446 0.070356 0.609817    0.071539     0.606290  0.001182 -0.003527
   42 0.996313 0.241773 0.063939 0.619568    0.064518     0.618660  0.000579 -0.000908
  100 0.938601 0.242378 0.069005 0.610487    0.067807     0.610518 -0.001198  0.000031
 2026 0.928429 0.238931 0.062095 0.612474    0.063979     0.610212  0.001884 -0.002262
 7777 0.963661 0.235253 0.062635 0.616367    0.064379     0.614962  0.001744 -0.001405

===== 평균 =====
AP_70          0.065606
WLL_70         0.613743
AP_120_cal     0.066444
WLL_120_cal    0.612128
AP_gain        0.000838
WLL_gain      -0.001614
dtype: float64


- 5번 나눠서 확인 시, 평균적으로 AP는 +0.000838 상승, WLL은 -0.001614 감소(개선)

- 120 iteration +calibration 채택

- 원본 117개 피처 LightGBM -> 120 tree -> probability calibration이 가장 좋은 방향

- seq_len, count/frequency, interaction, seq 첫/마지막 토큰 제거 - 제외

In [ ]:
# 200만 행에서 검증
#200만 행 랜덤 샘플 생성

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

pf = pq.ParquetFile("/content/train.parquet")

# seq 제외
cols = [
    c for c in pf.schema_arrow.names
    if c != "seq"
]

rng = np.random.default_rng(42)

chunks = []
collected = 0
TARGET_ROWS = 2_000_000

for i, batch in enumerate(
    pf.iter_batches(
        columns=cols,
        batch_size=200_000
    )
):
    tmp = batch.to_pandas()

    # 전체의 약 20% 랜덤 추출
    mask = rng.random(len(tmp)) < 0.20
    tmp = tmp.loc[mask].copy()

    chunks.append(tmp)
    collected += len(tmp)

    del batch, tmp
    gc.collect()

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches / {collected:,} rows")

    if collected >= TARGET_ROWS:
        break

df2m = pd.concat(chunks, ignore_index=True)
df2m = df2m.iloc[:TARGET_ROWS].copy()

del chunks
gc.collect()

print("\n완료")
print("shape:", df2m.shape)
print("CTR:", df2m["clicked"].mean())
print(
    f"메모리: {df2m.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

10 batches / 400,015 rows
20 batches / 800,195 rows
30 batches / 1,199,600 rows
40 batches / 1,599,864 rows
50 batches / 2,000,199 rows

완료
shape: (2000000, 118)
CTR: 0.018913
메모리: 1.32 GB


In [ ]:
# categorical 처리 + 80:20 train/validation 분리
from sklearn.model_selection import train_test_split
import numpy as np

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

for c in cat_cols:
    df2m[c] = (
        df2m[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

train_idx2m, valid_idx2m = train_test_split(
    np.arange(len(df2m)),
    test_size=0.20,
    random_state=42,
    stratify=df2m["clicked"]
)

features2m = [
    c for c in df2m.columns
    if c != "clicked"
]

X_train2m = df2m.loc[train_idx2m, features2m]
X_valid2m = df2m.loc[valid_idx2m, features2m]

y_train2m = df2m.loc[train_idx2m, "clicked"]
y_valid2m = df2m.loc[valid_idx2m, "clicked"]

print("Train:", X_train2m.shape)
print("Valid:", X_valid2m.shape)
print("Train CTR:", y_train2m.mean())
print("Valid CTR:", y_valid2m.mean())

Train: (1600000, 117)
Valid: (400000, 117)
Train CTR: 0.018913125
Valid CTR: 0.0189125


In [ ]:
# 200만 행 데이터로 LightGBM 모델 학습
from lightgbm import LGBMClassifier

model2m = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=160,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model2m.fit(
    X_train2m,
    y_train2m,
    categorical_feature=cat_cols
)

print("LightGBM 학습 완료")

[LightGBM] [Info] Number of positive: 30261, number of negative: 1569739
[LightGBM] [Info] Total Bins 17683
[LightGBM] [Info] Number of data points in the train set: 1600000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
LightGBM 학습 완료


In [ ]:
# iteration별 AP/WLL 확인
from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

def ap_wll(y, pred):
    y = np.asarray(y)
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y, pred)

    pos_loss = -np.mean(np.log(pred[y == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss

    return ap, wll

y2m = y_valid2m.to_numpy()
rows = []

for n in [50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160]:

    pred = model2m.predict_proba(
        X_valid2m,
        num_iteration=n
    )[:, 1]

    ap, wll = ap_wll(y2m, pred)

    rows.append({
        "iter": n,
        "AP": ap,
        "WLL": wll
    })

result2m = pd.DataFrame(rows)
print(result2m.to_string(index=False))

 iter       AP      WLL
   50 0.068172 0.612249
   60 0.068902 0.609841
   70 0.070128 0.608294
   80 0.071214 0.607173
   90 0.071866 0.606591
  100 0.072314 0.606102
  110 0.072664 0.605676
  120 0.073070 0.605685
  130 0.073400 0.605672
  140 0.073646 0.605731
  150 0.073884 0.605596
  160 0.073985 0.605604


- 200만 행에서의 패턴이 50만 행에서보다 좋은 결과

- 50만 행에서는, iter 70 이후 WLL 다시 나빠짐

- 200만 행에서는, A 160까지 계속 상승, WLL도 150~160까지 거의 계속 좋아짐

- 160 iter에서 가장 좋은 결과

In [ ]:
# 160 iter 예측을 calibration 진행 시 WLL값이 더 내려가는지 확인

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid2m.to_numpy()

# 150 / 160 iteration 예측
p150 = model2m.predict_proba(
    X_valid2m,
    num_iteration=150
)[:, 1]

p160 = model2m.predict_proba(
    X_valid2m,
    num_iteration=160
)[:, 1]

p150 = np.clip(p150, 1e-8, 1 - 1e-8)
p160 = np.clip(p160, 1e-8, 1 - 1e-8)

z160 = np.log(p160 / (1 - p160))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll


rows = []

for seed in [1, 42, 100, 2026, 7777]:

    idx = np.arange(len(y))

    cal_idx, eval_idx = train_test_split(
        idx,
        test_size=0.5,
        random_state=seed,
        stratify=y
    )

    def loss(theta):
        a, b = theta

        q = sigmoid(
            a * z160[cal_idx] + b
        )

        yy = y[cal_idx]

        pos = -np.mean(np.log(q[yy == 1]))
        neg = -np.mean(np.log(1 - q[yy == 0]))

        return 0.5 * pos + 0.5 * neg

    res = minimize(
        loss,
        x0=[1.0, 0.0],
        method="Nelder-Mead"
    )

    a, b = res.x

    y_eval = y[eval_idx]

    ap150, wll150 = evaluate(
        y_eval,
        p150[eval_idx]
    )

    ap160_raw, wll160_raw = evaluate(
        y_eval,
        p160[eval_idx]
    )

    p160_cal = sigmoid(
        a * z160[eval_idx] + b
    )

    ap160_cal, wll160_cal = evaluate(
        y_eval,
        p160_cal
    )

    rows.append({
        "seed": seed,
        "a": a,
        "b": b,

        "AP_150": ap150,
        "WLL_150": wll150,

        "AP_160_raw": ap160_raw,
        "WLL_160_raw": wll160_raw,

        "AP_160_cal": ap160_cal,
        "WLL_160_cal": wll160_cal
    })

check2m = pd.DataFrame(rows)

print(check2m.to_string(index=False))

print("\n===== 평균 =====")
print(
    check2m[
        [
            "AP_150", "WLL_150",
            "AP_160_raw", "WLL_160_raw",
            "AP_160_cal", "WLL_160_cal"
        ]
    ].mean()
)

 seed        a        b   AP_150  WLL_150  AP_160_raw  WLL_160_raw  AP_160_cal  WLL_160_cal
    1 1.032495 0.091266 0.074363 0.606405    0.074502     0.606343    0.074502     0.605516
   42 1.057752 0.083771 0.071670 0.610161    0.071739     0.610245    0.071739     0.609555
  100 0.997889 0.089669 0.069919 0.602899    0.070023     0.602990    0.070023     0.602222
 2026 1.013686 0.091607 0.072742 0.604136    0.072739     0.604133    0.072739     0.603301
 7777 1.018576 0.091910 0.074287 0.604791    0.074372     0.604901    0.074372     0.604064

===== 평균 =====
AP_150         0.072596
WLL_150        0.605678
AP_160_raw     0.072675
WLL_160_raw    0.605722
AP_160_cal     0.072675
WLL_160_cal    0.604932
dtype: float64


- AP가 160iter에서 상승 추세 -> 200~260 tree까지 확인

In [ ]:
# 새로 260까지 학습

from lightgbm import LGBMClassifier

model2m_260 = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=260,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model2m_260.fit(
    X_train2m,
    y_train2m,
    categorical_feature=cat_cols
)

print("260 tree 학습 완료")

[LightGBM] [Info] Number of positive: 30261, number of negative: 1569739
[LightGBM] [Info] Total Bins 17683
[LightGBM] [Info] Number of data points in the train set: 1600000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
260 tree 학습 완료


In [ ]:
rows = []

for n in [140, 150, 160, 180, 200, 220, 240, 260]:

    pred = model2m_260.predict_proba(
        X_valid2m,
        num_iteration=n
    )[:, 1]

    ap, wll = ap_wll(y_valid2m.to_numpy(), pred)

    rows.append({
        "iter": n,
        "AP": ap,
        "WLL": wll
    })

result260 = pd.DataFrame(rows)
print(result260.to_string(index=False))

 iter       AP      WLL
  140 0.073646 0.605731
  150 0.073884 0.605596
  160 0.073985 0.605604
  180 0.074128 0.605779
  200 0.074048 0.606085
  220 0.074057 0.606143
  240 0.074131 0.606427
  260 0.074130 0.606761


- 180 전후가 가장 좋은 결과

In [ ]:
# 180 iter 기준 calibration
# 180 주변 촘촘한 확인

rows = []

for n in range(150, 201, 5):

    pred = model2m_260.predict_proba(
        X_valid2m,
        num_iteration=n
    )[:, 1]

    ap, wll = ap_wll(
        y_valid2m.to_numpy(),
        pred
    )

    rows.append({
        "iter": n,
        "AP": ap,
        "WLL": wll
    })

fine_result = pd.DataFrame(rows)

print(fine_result.to_string(index=False))

 iter       AP      WLL
  150 0.073884 0.605596
  155 0.073997 0.605649
  160 0.073985 0.605604
  165 0.073982 0.605661
  170 0.074032 0.605735
  175 0.074109 0.605717
  180 0.074128 0.605779
  185 0.074028 0.605873
  190 0.074048 0.605864
  195 0.074043 0.605976
  200 0.074048 0.606085


- 최적 구간: 175~180 근처

In [ ]:
# 150/175/180 세 모델에 calibration 적용하여 seed 평균 비교

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid2m.to_numpy()

candidate_iters = [150, 175, 180]

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll


rows = []

for n in candidate_iters:

    p = model2m_260.predict_proba(
        X_valid2m,
        num_iteration=n
    )[:, 1]

    p = np.clip(p, 1e-8, 1 - 1e-8)
    z = np.log(p / (1 - p))

    for seed in [1, 42, 100, 2026, 7777]:

        idx = np.arange(len(y))

        cal_idx, eval_idx = train_test_split(
            idx,
            test_size=0.5,
            random_state=seed,
            stratify=y
        )

        def loss(theta):
            a, b = theta

            q = sigmoid(
                a * z[cal_idx] + b
            )

            yy = y[cal_idx]

            pos = -np.mean(np.log(q[yy == 1]))
            neg = -np.mean(np.log(1 - q[yy == 0]))

            return 0.5 * pos + 0.5 * neg

        res = minimize(
            loss,
            x0=[1.0, 0.0],
            method="Nelder-Mead"
        )

        a, b = res.x

        y_eval = y[eval_idx]

        p_raw = p[eval_idx]

        p_cal = sigmoid(
            a * z[eval_idx] + b
        )

        ap_raw, wll_raw = evaluate(
            y_eval,
            p_raw
        )

        ap_cal, wll_cal = evaluate(
            y_eval,
            p_cal
        )

        rows.append({
            "iter": n,
            "seed": seed,
            "a": a,
            "b": b,
            "AP_raw": ap_raw,
            "WLL_raw": wll_raw,
            "AP_cal": ap_cal,
            "WLL_cal": wll_cal
        })

compare_cal = pd.DataFrame(rows)

print("===== seed별 결과 =====")
print(compare_cal.to_string(index=False))

print("\n===== iteration별 평균 =====")
print(
    compare_cal.groupby("iter")[
        ["AP_raw", "WLL_raw", "AP_cal", "WLL_cal"]
    ].mean()
)

===== seed별 결과 =====
 iter  seed        a        b   AP_raw  WLL_raw   AP_cal  WLL_cal
  150     1 1.037258 0.086225 0.074363 0.606405 0.074363 0.605661
  150    42 1.061514 0.078621 0.071670 0.610161 0.071670 0.609569
  150   100 1.001124 0.084952 0.069919 0.602899 0.069919 0.602201
  150  2026 1.017921 0.086753 0.072742 0.604136 0.072742 0.603375
  150  7777 1.022101 0.087209 0.074287 0.604791 0.074287 0.604031
  175     1 1.025188 0.098702 0.074761 0.606286 0.074761 0.605327
  175    42 1.050946 0.091227 0.072014 0.610333 0.072014 0.609470
  175   100 0.990665 0.096620 0.070055 0.602967 0.070055 0.602108
  175  2026 1.006379 0.098702 0.072952 0.604118 0.072952 0.603178
  175  7777 1.011742 0.099044 0.074404 0.604950 0.074404 0.603996
  180     1 1.023126 0.101122 0.074706 0.606332 0.074706 0.605327
  180    42 1.048366 0.093845 0.072075 0.610313 0.072075 0.609384
  180   100 0.989166 0.098827 0.070082 0.603057 0.070082 0.602157
  180  2026 1.004047 0.101036 0.072918 0.604098 0.07291

- 최적의 조헙: 180 iter+calibration

In [ ]:
# 500만 행에서의 검사
# 500만 행 랜덤 샘플
# categorical 처리 & 80:20 split
# train5m, valid5m 생

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

pf = pq.ParquetFile("/content/train.parquet")

cols = [
    c for c in pf.schema_arrow.names
    if c != "seq"
]

rng = np.random.default_rng(42)

chunks = []
collected = 0
TARGET_ROWS = 5_000_000
SAMPLE_RATE = 0.47

for i, batch in enumerate(
    pf.iter_batches(
        columns=cols,
        batch_size=200_000
    )
):
    tmp = batch.to_pandas()

    mask = rng.random(len(tmp)) < SAMPLE_RATE
    tmp = tmp.loc[mask].copy()

    if len(tmp) > 0:
        chunks.append(tmp)
        collected += len(tmp)

    del batch, tmp
    gc.collect()

    if (i + 1) % 5 == 0:
        print(f"{i+1} batches / {collected:,} rows")

    if collected >= TARGET_ROWS:
        break

df5m = pd.concat(chunks, ignore_index=True)
df5m = df5m.iloc[:TARGET_ROWS].copy()

del chunks
gc.collect()

print("완료")
print("shape:", df5m.shape)
print("CTR:", df5m["clicked"].mean())
print(
    "메모리:",
    round(df5m.memory_usage(deep=True).sum() / 1024**3, 2),
    "GB"
)

5 batches / 469,897 rows
10 batches / 940,299 rows
15 batches / 1,410,630 rows
20 batches / 1,880,912 rows
25 batches / 2,350,535 rows
30 batches / 2,820,427 rows
35 batches / 3,291,557 rows
40 batches / 3,761,706 rows
45 batches / 4,232,418 rows
50 batches / 4,702,829 rows
완료
shape: (5000000, 118)
CTR: 0.0190156
메모리: 3.3 GB


In [ ]:
import numpy as np
import gc
from sklearn.model_selection import train_test_split

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

for c in cat_cols:
    df5m[c] = (
        df5m[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

idx = np.arange(len(df5m))

train_idx5m, valid_idx5m = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=df5m["clicked"]
)

train5m = df5m.iloc[train_idx5m].copy()
valid5m = df5m.iloc[valid_idx5m].copy()

del df5m, train_idx5m, valid_idx5m, idx
gc.collect()

y_train5m = train5m.pop("clicked")
y_valid5m = valid5m.pop("clicked")

X_train5m = train5m
X_valid5m = valid5m

print("Train:", X_train5m.shape)
print("Valid:", X_valid5m.shape)

Train: (4000000, 117)
Valid: (1000000, 117)


In [ ]:
# 500만 행 실험용 LightGBM 학습
# 400만 행 train 데이터로 320 tree까지 한 번 학습

from lightgbm import LGBMClassifier

model5m = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=320,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model5m.fit(
    X_train5m,
    y_train5m,
    categorical_feature=cat_cols
)

print("500만 행 LightGBM 학습 완료")


[LightGBM] [Info] Number of positive: 76062, number of negative: 3923938
[LightGBM] [Info] Total Bins 17684
[LightGBM] [Info] Number of data points in the train set: 4000000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
500만 행 LightGBM 학습 완료


In [ ]:
# 500만 행에서 최적 iteration 확인

from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

def ap_wll(y, pred):
    y = np.asarray(y)
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y, pred)

    pos_loss = -np.mean(np.log(pred[y == 1]))
    neg_loss = -np.mean(np.log(1 - pred[y == 0]))

    wll = 0.5 * pos_loss + 0.5 * neg_loss

    return ap, wll


y5m = y_valid5m.to_numpy()
rows = []

for n in [
    140, 160, 180, 200,
    220, 240, 260, 280,
    300, 320
]:

    pred = model5m.predict_proba(
        X_valid5m,
        num_iteration=n
    )[:, 1]

    ap, wll = ap_wll(y5m, pred)

    rows.append({
        "iter": n,
        "AP": ap,
        "WLL": wll
    })

result5m = pd.DataFrame(rows)

print(result5m.to_string(index=False))

 iter       AP      WLL
  140 0.076378 0.599772
  160 0.076814 0.599469
  180 0.077148 0.599373
  200 0.077361 0.599291
  220 0.077491 0.599315
  240 0.077657 0.599282
  260 0.077792 0.599315
  280 0.078028 0.599350
  300 0.078216 0.599444
  320 0.078312 0.599491


- 240 iter에서 WLL 최저 = 0.599282, AP는 계속 상승해서 0.078312

- 240->320에서 AP +0.000655, WLL 손해 +0.000209

In [ ]:
# 240/280/320 calibration

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid5m.to_numpy()

candidate_iters = [240, 280, 320]

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll


rows = []

for n in candidate_iters:

    p = model5m.predict_proba(
        X_valid5m,
        num_iteration=n
    )[:, 1]

    p = np.clip(p, 1e-8, 1 - 1e-8)
    z = np.log(p / (1 - p))

    for seed in [1, 42, 100, 2026, 7777]:

        idx = np.arange(len(y))

        cal_idx, eval_idx = train_test_split(
            idx,
            test_size=0.5,
            random_state=seed,
            stratify=y
        )

        # log_a를 최적화해서 a > 0 보장
        def loss(theta):
            log_a, b = theta
            a = np.exp(log_a)

            q = sigmoid(
                a * z[cal_idx] + b
            )

            yy = y[cal_idx]

            pos = -np.mean(np.log(q[yy == 1]))
            neg = -np.mean(np.log(1 - q[yy == 0]))

            return 0.5 * pos + 0.5 * neg

        res = minimize(
            loss,
            x0=[0.0, 0.0],
            method="Nelder-Mead"
        )

        log_a, b = res.x
        a = np.exp(log_a)

        y_eval = y[eval_idx]

        p_raw = p[eval_idx]
        p_cal = sigmoid(
            a * z[eval_idx] + b
        )

        ap_raw, wll_raw = evaluate(
            y_eval,
            p_raw
        )

        ap_cal, wll_cal = evaluate(
            y_eval,
            p_cal
        )

        rows.append({
            "iter": n,
            "seed": seed,
            "a": a,
            "b": b,
            "AP_raw": ap_raw,
            "WLL_raw": wll_raw,
            "AP_cal": ap_cal,
            "WLL_cal": wll_cal
        })

compare5m = pd.DataFrame(rows)

print("===== seed별 결과 =====")
print(compare5m.to_string(index=False))

print("\n===== iteration별 평균 =====")
print(
    compare5m.groupby("iter")[
        ["AP_raw", "WLL_raw", "AP_cal", "WLL_cal"]
    ].mean()
)

===== seed별 결과 =====
 iter  seed        a        b   AP_raw  WLL_raw   AP_cal  WLL_cal
  240     1 1.027491 0.055207 0.078574 0.598127 0.078574 0.597768
  240    42 1.029822 0.055851 0.078437 0.598647 0.078437 0.598298
  240   100 1.042464 0.052240 0.078215 0.600148 0.078215 0.599822
  240  2026 1.039874 0.052849 0.077238 0.600014 0.077238 0.599675
  240  7777 1.033702 0.051974 0.076999 0.599836 0.076999 0.599458
  280     1 1.020379 0.062933 0.079017 0.598151 0.079017 0.597736
  280    42 1.022564 0.063523 0.078791 0.598672 0.078791 0.598265
  280   100 1.036842 0.060012 0.078693 0.600339 0.078693 0.599945
  280  2026 1.033039 0.060522 0.077487 0.600132 0.077487 0.599720
  280  7777 1.027841 0.059678 0.077472 0.600004 0.077472 0.599561
  320     1 1.011496 0.070592 0.079336 0.598191 0.079336 0.597706
  320    42 1.015137 0.071214 0.079073 0.598884 0.079073 0.598403
  320   100 1.028196 0.067913 0.079067 0.600429 0.079067 0.599937
  320  2026 1.025233 0.068167 0.077693 0.600368 0.07769

- 240->320, AP는 +0.000674,
WLL은 +0.000098 (나빠짐, 매우 작은 손해)

- 320iter이 최적의 상태

In [ ]:
# 400~480 iter 확인

from lightgbm import LGBMClassifier

model5m_480 = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=480,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

model5m_480.fit(
    X_train5m,
    y_train5m,
    categorical_feature=cat_cols
)

print("480 tree 학습 완료")

[LightGBM] [Info] Number of positive: 76062, number of negative: 3923938
[LightGBM] [Info] Total Bins 17684
[LightGBM] [Info] Number of data points in the train set: 4000000, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
480 tree 학습 완료


In [ ]:
rows = []

for n in [
    280, 320, 340, 360, 380,
    400, 420, 440, 460, 480
]:

    pred = model5m_480.predict_proba(
        X_valid5m,
        num_iteration=n
    )[:, 1]

    ap, wll = ap_wll(
        y_valid5m.to_numpy(),
        pred
    )

    rows.append({
        "iter": n,
        "AP": ap,
        "WLL": wll
    })

result480 = pd.DataFrame(rows)

print(result480.to_string(index=False))

 iter       AP      WLL
  280 0.078028 0.599350
  320 0.078312 0.599491
  340 0.078376 0.599635
  360 0.078369 0.599722
  380 0.078427 0.599748
  400 0.078588 0.599895
  420 0.078705 0.600034
  440 0.078826 0.600181
  460 0.078813 0.600287
  480 0.078907 0.600380


- AP는 480까지 계속 상승, WLL은 320 이후 계속 안좋아짐.

In [ ]:
# 320/400/440/480 calibration 후 비교

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

y = y_valid5m.to_numpy()

candidate_iters = [320, 400, 440, 480]

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def evaluate(y_true, pred):
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y_true, pred)

    pos = -np.mean(np.log(pred[y_true == 1]))
    neg = -np.mean(np.log(1 - pred[y_true == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll

rows = []

for n in candidate_iters:

    p = model5m_480.predict_proba(
        X_valid5m,
        num_iteration=n
    )[:, 1]

    p = np.clip(p, 1e-8, 1 - 1e-8)
    z = np.log(p / (1 - p))

    for seed in [1, 42, 100, 2026, 7777]:

        idx = np.arange(len(y))

        cal_idx, eval_idx = train_test_split(
            idx,
            test_size=0.5,
            random_state=seed,
            stratify=y
        )

        def loss(theta):
            log_a, b = theta
            a = np.exp(log_a)

            q = sigmoid(
                a * z[cal_idx] + b
            )

            yy = y[cal_idx]

            pos = -np.mean(np.log(q[yy == 1]))
            neg = -np.mean(np.log(1 - q[yy == 0]))

            return 0.5 * pos + 0.5 * neg

        res = minimize(
            loss,
            x0=[0.0, 0.0],
            method="Nelder-Mead"
        )

        log_a, b = res.x
        a = np.exp(log_a)

        y_eval = y[eval_idx]

        p_raw = p[eval_idx]
        p_cal = sigmoid(
            a * z[eval_idx] + b
        )

        ap_raw, wll_raw = evaluate(
            y_eval,
            p_raw
        )

        ap_cal, wll_cal = evaluate(
            y_eval,
            p_cal
        )

        rows.append({
            "iter": n,
            "seed": seed,
            "a": a,
            "b": b,
            "AP_raw": ap_raw,
            "WLL_raw": wll_raw,
            "AP_cal": ap_cal,
            "WLL_cal": wll_cal
        })

compare_final5m = pd.DataFrame(rows)

print("===== iteration별 평균 =====")
print(
    compare_final5m.groupby("iter")[
        ["AP_raw", "WLL_raw", "AP_cal", "WLL_cal"]
    ].mean()
)

print("\n===== calibration parameter 평균 =====")
print(
    compare_final5m.groupby("iter")[["a", "b"]].mean()
)

===== iteration별 평균 =====
        AP_raw   WLL_raw    AP_cal   WLL_cal
iter                                        
320   0.078567  0.599601  0.078567  0.599102
400   0.078714  0.600031  0.078714  0.599288
440   0.078939  0.600338  0.078939  0.599447
480   0.079025  0.600548  0.079025  0.599496

===== calibration parameter 평균 =====
             a         b
iter                    
320   1.019844  0.069041
400   1.009392  0.084797
440   1.003299  0.092511
480   0.998758  0.099971


- 440->480 시, AP +0.000086, WLL +0.000049

- 최종 후보: 80 iter + calibration

In [ ]:
# 480 tree 모델에서, 최종 확률 보정값 a,b 정하기

import numpy as np
from scipy.optimize import minimize

y = y_valid5m.to_numpy()

# 480 iteration validation prediction
p480 = model5m_480.predict_proba(
    X_valid5m,
    num_iteration=480
)[:, 1]

p480 = np.clip(p480, 1e-8, 1 - 1e-8)
z480 = np.log(p480 / (1 - p480))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def loss(theta):
    log_a, b = theta
    a = np.exp(log_a)

    q = sigmoid(a * z480 + b)

    pos = -np.mean(np.log(q[y == 1]))
    neg = -np.mean(np.log(1 - q[y == 0]))

    return 0.5 * pos + 0.5 * neg

res = minimize(
    loss,
    x0=[0.0, 0.10],
    method="Nelder-Mead"
)

final_a = np.exp(res.x[0])
final_b = res.x[1]

p480_cal = sigmoid(
    final_a * z480 + final_b
)

ap_final, wll_final = ap_wll(
    y,
    p480_cal
)

print("final a:", final_a)
print("final b:", final_b)
print("AP :", ap_final)
print("WLL:", wll_final)

final a: 0.9976650979550782
final b: 0.10057041168212894
AP : 0.07890749125605548
WLL: 0.5993305149140713


- LightGBM의 log-odds를 약 +0.10만큼 이동시키는 보정

- 최종 후보: seq 제외 원본 117개 피처 -> LightGBM 480 trees -> logit calibration(a=0.9977, b=0.10057)


In [ ]:
# RAM 한계 -> 전체 1,070만 행 한 번에 LGBM 수행 X
# 전체 Train 약 1070만 행을 랜덤하게
# A 약 500만 / B 약 500만 / calibration 약 70만으로 나누기
# A 학습용 약 498만->LightGBM A
# B 학습용 약 498만->LightGBM B
# Holdout 약 75만->A,B가 둘 다 예측 (검증+calibration용, A,B 둘 다 한 번도 못 본 데이터 남겨두기)
# 추후 A만 사용, B만 사용, 평균내기 중 비교

# 전체 train에서 무작위 약 46.5%, 약 498만 행을 골라 480-tree 모델 A 생성
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc
import joblib
import json

from lightgbm import LGBMClassifier

TRAIN_PATH = "/content/train.parquet"

pf = pq.ParquetFile(TRAIN_PATH)

# seq 제외
feature_cols = [
    c for c in pf.schema_arrow.names
    if c not in ["seq", "clicked"]
]

read_cols = feature_cols + ["clicked"]

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

# 항상 같은 랜덤 분할이 나오도록 seed 고정
rng = np.random.default_rng(2026)

chunks = []
count = 0

for i, batch in enumerate(
    pf.iter_batches(
        columns=read_cols,
        batch_size=200_000
    )
):
    tmp = batch.to_pandas()

    # A 약 46.5%
    r = rng.random(len(tmp))
    tmp = tmp.loc[r < 0.465].copy()

    if len(tmp) > 0:
        chunks.append(tmp)
        count += len(tmp)

    del batch, tmp, r
    gc.collect()

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches / A: {count:,} rows")


dfA = pd.concat(chunks, ignore_index=True)

del chunks
gc.collect()

print("\nA 데이터 완료")
print("shape:", dfA.shape)
print("CTR:", dfA["clicked"].mean())
print(
    "메모리:",
    round(dfA.memory_usage(deep=True).sum() / 1024**3, 2),
    "GB"
)

# categorical 처리
cat_levels_A = {}

for c in cat_cols:
    dfA[c] = (
        dfA[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

    cat_levels_A[c] = dfA[c].cat.categories.astype(str).tolist()


yA = dfA.pop("clicked")
XA = dfA

print("\n모델 A 학습 시작")

modelA = LGBMClassifier(
    objective="binary",
    class_weight="balanced",

    n_estimators=480,
    learning_rate=0.05,
    num_leaves=63,

    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

modelA.fit(
    XA,
    yA,
    categorical_feature=cat_cols
)

print("모델 A 학습 완료")

# 런타임 초기화 대비 저장
joblib.dump(
    modelA,
    "/content/modelA.pkl"
)

with open(
    "/content/cat_levels_A.json",
    "w"
) as f:
    json.dump(cat_levels_A, f)

with open(
    "/content/feature_cols.json",
    "w"
) as f:
    json.dump(feature_cols, f)

print("modelA 저장 완료")

10 batches / A: 930,383 rows
20 batches / A: 1,861,720 rows
30 batches / A: 2,791,512 rows
40 batches / A: 3,721,262 rows
50 batches / A: 4,650,676 rows

A 데이터 완료
shape: (4978317, 118)
CTR: 0.019072108104003824
메모리: 3.28 GB

모델 A 학습 시작
[LightGBM] [Info] Number of positive: 94947, number of negative: 4883370
[LightGBM] [Info] Total Bins 17676
[LightGBM] [Info] Number of data points in the train set: 4978317, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
모델 A 학습 완료
modelA 저장 완료


In [ ]:
# 모델 A로 test 152만 행 예측
# pred_A,npy로 저장
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

TEST_PATH = "/content/test.parquet"

pf_test = pq.ParquetFile(TEST_PATH)

pred_A_parts = []
total = 0

for i, batch in enumerate(
    pf_test.iter_batches(
        columns=feature_cols,
        batch_size=100_000
    )
):
    tmp = batch.to_pandas()

    # train A와 완전히 같은 categorical 기준 적용
    for c in cat_cols:
        tmp[c] = pd.Categorical(
            tmp[c].astype("string").fillna("MISSING"),
            categories=cat_levels_A[c]
        )

    pred = modelA.predict_proba(tmp)[:, 1]

    pred_A_parts.append(pred.astype("float32"))
    total += len(pred)

    del batch, tmp, pred
    gc.collect()

    print(f"{total:,} rows 예측 완료")

pred_A = np.concatenate(pred_A_parts)

np.save("/content/pred_A.npy", pred_A)

del pred_A_parts
gc.collect()

print("\n모델 A test 예측 완료")
print("예측 개수:", len(pred_A))
print("평균 예측값:", pred_A.mean())
print("저장 위치: /content/pred_A.npy")

100,000 rows 예측 완료
200,000 rows 예측 완료
300,000 rows 예측 완료
400,000 rows 예측 완료
500,000 rows 예측 완료
600,000 rows 예측 완료
700,000 rows 예측 완료
800,000 rows 예측 완료
900,000 rows 예측 완료
1,000,000 rows 예측 완료
1,100,000 rows 예측 완료
1,200,000 rows 예측 완료
1,300,000 rows 예측 완료
1,400,000 rows 예측 완료
1,500,000 rows 예측 완료
1,527,298 rows 예측 완료

모델 A test 예측 완료 ✅
예측 개수: 1527298
평균 예측값: 0.393416
저장 위치: /content/pred_A.npy


In [ ]:
# 모델 A와 겹치지 않는 다른 약 498만 행으로 모델 B 생성
# A의 난수: r<0.465, B의 난수: 0.465<=r<0.93
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc
import joblib
import json

from lightgbm import LGBMClassifier

TRAIN_PATH = "/content/train.parquet"

pf = pq.ParquetFile(TRAIN_PATH)

feature_cols = [
    c for c in pf.schema_arrow.names
    if c not in ["seq", "clicked"]
]

read_cols = feature_cols + ["clicked"]

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

# A와 반드시 같은 seed
rng = np.random.default_rng(2026)

chunks = []
count = 0

for i, batch in enumerate(
    pf.iter_batches(
        columns=read_cols,
        batch_size=200_000
    )
):
    tmp = batch.to_pandas()

    # A: r < 0.465
    # B: 0.465 <= r < 0.93
    # calibration: r >= 0.93 (나중에 사용)
    r = rng.random(len(tmp))

    mask = (r >= 0.465) & (r < 0.93)
    tmp = tmp.loc[mask].copy()

    if len(tmp) > 0:
        chunks.append(tmp)
        count += len(tmp)

    del batch, tmp, r, mask
    gc.collect()

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches / B: {count:,} rows")


dfB = pd.concat(chunks, ignore_index=True)

del chunks
gc.collect()

print("\nB 데이터 완료")
print("shape:", dfB.shape)
print("CTR:", dfB["clicked"].mean())
print(
    "메모리:",
    round(dfB.memory_usage(deep=True).sum() / 1024**3, 2),
    "GB"
)

cat_levels_B = {}

for c in cat_cols:
    dfB[c] = (
        dfB[c]
        .astype("string")
        .fillna("MISSING")
        .astype("category")
    )

    cat_levels_B[c] = dfB[c].cat.categories.astype(str).tolist()

yB = dfB.pop("clicked")
XB = dfB

print("\n모델 B 학습 시작")

modelB = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    n_estimators=480,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=777,
    n_jobs=-1,
    force_col_wise=True
)

modelB.fit(
    XB,
    yB,
    categorical_feature=cat_cols
)

print("모델 B 학습 완료")

joblib.dump(modelB, "/content/modelB.pkl")

with open("/content/cat_levels_B.json", "w") as f:
    json.dump(cat_levels_B, f)

print("modelB 저장 완료")

10 batches / B: 929,910 rows
20 batches / B: 1,859,139 rows
30 batches / B: 2,789,128 rows
40 batches / B: 3,719,836 rows
50 batches / B: 4,650,131 rows

B 데이터 완료
shape: (4977014, 118)
CTR: 0.01906203197338806
메모리: 3.28 GB

모델 B 학습 시작
[LightGBM] [Info] Number of positive: 94872, number of negative: 4882142
[LightGBM] [Info] Total Bins 17687
[LightGBM] [Info] Number of data points in the train set: 4977014, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
모델 B 학습 완료
modelB 저장 완료


In [ ]:
# 모델 B test 예측 저장
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

TEST_PATH = "/content/test.parquet"

pf_test = pq.ParquetFile(TEST_PATH)

pred_B_parts = []
total = 0

for i, batch in enumerate(
    pf_test.iter_batches(
        columns=feature_cols,
        batch_size=100_000
    )
):
    tmp = batch.to_pandas()

    # 모델 B가 학습할 때 사용한 category 기준 그대로 적용
    for c in cat_cols:
        tmp[c] = pd.Categorical(
            tmp[c].astype("string").fillna("MISSING"),
            categories=cat_levels_B[c]
        )

    pred = modelB.predict_proba(tmp)[:, 1]

    pred_B_parts.append(pred.astype("float32"))
    total += len(pred)

    del batch, tmp, pred
    gc.collect()

    print(f"{total:,} rows 예측 완료")

pred_B = np.concatenate(pred_B_parts)

np.save("/content/pred_B.npy", pred_B)

del pred_B_parts
gc.collect()

print("\n모델 B test 예측 완료")
print("예측 개수:", len(pred_B))
print("평균 예측값:", pred_B.mean())
print("저장 위치: /content/pred_B.npy")

100,000 rows 예측 완료
200,000 rows 예측 완료
300,000 rows 예측 완료
400,000 rows 예측 완료
500,000 rows 예측 완료
600,000 rows 예측 완료
700,000 rows 예측 완료
800,000 rows 예측 완료
900,000 rows 예측 완료
1,000,000 rows 예측 완료
1,100,000 rows 예측 완료
1,200,000 rows 예측 완료
1,300,000 rows 예측 완료
1,400,000 rows 예측 완료
1,500,000 rows 예측 완료
1,527,298 rows 예측 완료

모델 B test 예측 완료
예측 개수: 1527298
평균 예측값: 0.36548492
저장 위치: /content/pred_B.npy


- A 학습: 4,978,317행

- B 학습: 4,977,014행

- A, B  합쳐 약 995만 행 학습에만 사용


In [ ]:
# r>=0.93 시, A,B 학습 시 사용 X 약 7% 데이터로(약 75만 행) 두 모델 평가
# 약 75만 행에 대해, A,B의 예측

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import joblib
import json
import gc

TRAIN_PATH = "/content/train.parquet"

# 저장했던 설정 다시 로드
with open("/content/feature_cols.json", "r") as f:
    feature_cols = json.load(f)

with open("/content/cat_levels_A.json", "r") as f:
    cat_levels_A = json.load(f)

with open("/content/cat_levels_B.json", "r") as f:
    cat_levels_B = json.load(f)

cat_cols = [
    "gender",
    "age_group",
    "inventory_id",
    "day_of_week",
    "hour"
]

modelA = joblib.load("/content/modelA.pkl")
modelB = joblib.load("/content/modelB.pkl")

pf = pq.ParquetFile(TRAIN_PATH)

read_cols = feature_cols + ["clicked"]

# A/B 만들 때와 정확히 같은 seed
rng = np.random.default_rng(2026)

y_parts = []
predA_parts = []
predB_parts = []

count = 0

for i, batch in enumerate(
    pf.iter_batches(
        columns=read_cols,
        batch_size=200_000
    )
):
    tmp = batch.to_pandas()

    # A/B 때와 똑같은 난수 생성
    r = rng.random(len(tmp))

    # A: r < 0.465
    # B: 0.465 <= r < 0.93
    # calibration/validation: r >= 0.93
    mask = r >= 0.93

    if mask.any():

        selected = tmp.loc[mask].copy()

        y_part = selected["clicked"].to_numpy()
        Xpart = selected[feature_cols].copy()

        # 모델 A용 categorical
        XA_part = Xpart.copy()

        for c in cat_cols:
            XA_part[c] = pd.Categorical(
                XA_part[c].astype("string").fillna("MISSING"),
                categories=cat_levels_A[c]
            )

        pA = modelA.predict_proba(XA_part)[:, 1]


        # 모델 B용 categorical
        XB_part = Xpart.copy()

        for c in cat_cols:
            XB_part[c] = pd.Categorical(
                XB_part[c].astype("string").fillna("MISSING"),
                categories=cat_levels_B[c]
            )

        pB = modelB.predict_proba(XB_part)[:, 1]


        y_parts.append(y_part.astype("int8"))
        predA_parts.append(pA.astype("float32"))
        predB_parts.append(pB.astype("float32"))

        count += len(y_part)

        del selected, Xpart, XA_part, XB_part
        del y_part, pA, pB

    del tmp, batch, r, mask
    gc.collect()

    if (i + 1) % 10 == 0:
        print(f"{i+1} batches / holdout: {count:,} rows")


y_hold = np.concatenate(y_parts)
pA_hold = np.concatenate(predA_parts)
pB_hold = np.concatenate(predB_parts)

del y_parts, predA_parts, predB_parts
gc.collect()

print("\nHoldout 완료")
print("행 수:", len(y_hold))
print("CTR:", y_hold.mean())

10 batches / holdout: 139,707 rows
20 batches / holdout: 279,141 rows
30 batches / holdout: 419,360 rows
40 batches / holdout: 558,902 rows
50 batches / holdout: 699,193 rows

Holdout 완료 ✅
행 수: 748848
CTR: 0.01917612118881268


In [ ]:
# A/B/A+B 앙상블 비교
# A+B RAW = A 예측과 B 예측을 50:50 평균
# A+B CAL = 그 평균값을 한 번 더 확률 보정
# A+B 확률 보정(Calibration) 결과도 확인

import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import average_precision_score

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def ap_wll(y, pred):
    y = np.asarray(y)
    pred = np.clip(pred, 1e-15, 1 - 1e-15)

    ap = average_precision_score(y, pred)

    pos = -np.mean(np.log(pred[y == 1]))
    neg = -np.mean(np.log(1 - pred[y == 0]))

    wll = 0.5 * pos + 0.5 * neg

    return ap, wll


# A/B 단독
apA, wllA = ap_wll(y_hold, pA_hold)
apB, wllB = ap_wll(y_hold, pB_hold)

# 50:50 앙상블
p_ens = (pA_hold + pB_hold) / 2

apEns, wllEns = ap_wll(
    y_hold,
    p_ens
)

# ensemble calibration
p_ens_clip = np.clip(
    p_ens,
    1e-8,
    1 - 1e-8
)

z = np.log(
    p_ens_clip / (1 - p_ens_clip)
)

def loss(theta):

    log_a, b = theta
    a = np.exp(log_a)

    q = sigmoid(
        a * z + b
    )

    pos = -np.mean(
        np.log(q[y_hold == 1])
    )

    neg = -np.mean(
        np.log(1 - q[y_hold == 0])
    )

    return 0.5 * pos + 0.5 * neg


res = minimize(
    loss,
    x0=[0.0, 0.10],
    method="Nelder-Mead"
)

final_a = np.exp(res.x[0])
final_b = res.x[1]

p_ens_cal = sigmoid(
    final_a * z + final_b
)

apCal, wllCal = ap_wll(
    y_hold,
    p_ens_cal
)

print("===== HOLDOUT RESULT =====")

print(
    f"A          AP={apA:.6f}  WLL={wllA:.6f}"
)

print(
    f"B          AP={apB:.6f}  WLL={wllB:.6f}"
)

print(
    f"A+B RAW    AP={apEns:.6f}  WLL={wllEns:.6f}"
)

print(
    f"A+B CAL    AP={apCal:.6f}  WLL={wllCal:.6f}"
)

print("\nfinal a:", final_a)
print("final b:", final_b)

===== HOLDOUT RESULT =====
A          AP=0.080944  WLL=0.597192
B          AP=0.081174  WLL=0.597318
A+B RAW    AP=0.082112  WLL=0.595850
A+B CAL    AP=0.082112  WLL=0.595120

final a: 1.047198239274272
final b: 0.07929810346213054


- A: AP 0.080944, WLL 0.597192
- B: AP 0.081174, WLL 0.597318
- A+B: AP 0.082112, WLL 0.595850
- A+B + calibration: AP 0.082112, WLL 0.595120

In [ ]:
# 제출 파일 생성
# 약 995만 행으로 두 모델 학습 + 약 75만 행으로 검증/calibration+test 전체 예측
import numpy as np
import pandas as pd

# 저장해둔 A/B test 예측 불러오기
pred_A = np.load("/content/pred_A.npy")
pred_B = np.load("/content/pred_B.npy")

print("A:", len(pred_A))
print("B:", len(pred_B))

# 50:50 ensemble
pred_ens = (pred_A + pred_B) / 2

# holdout에서 구한 calibration 계수
final_a = 1.047198239274272
final_b = 0.07929810346213054

# calibration
p = np.clip(pred_ens, 1e-8, 1 - 1e-8)
z = np.log(p / (1 - p))

pred_final = 1 / (
    1 + np.exp(
        -np.clip(final_a * z + final_b, -30, 30)
    )
)

print("최종 예측 개수:", len(pred_final))
print("최종 예측 평균:", pred_final.mean())
print("최소:", pred_final.min())
print("최대:", pred_final.max())

# sample submission 불러오기
submission = pd.read_csv("/content/sample_submission.csv")

print("submission shape:", submission.shape)

# clicked에 확률 넣기
submission["clicked"] = pred_final

# 저장
submission.to_csv(
    "/content/submission_AB_cal.csv",
    index=False
)

print("\n제출파일 생성 완료")
print("/content/submission_AB_cal.csv")
print(submission.head())

A: 1527298
B: 1527298
최종 예측 개수: 1527298
최종 예측 평균: 0.39130673
최소: 0.01620595
최대: 0.9914909
submission shape: (1527298, 2)

제출파일 생성 완료
/content/submission_AB_cal.csv
             ID   clicked
0  TEST_0000000  0.272084
1  TEST_0000001  0.321973
2  TEST_0000002  0.440823
3  TEST_0000003  0.482929
4  TEST_0000004  0.380353


In [ ]:
from google.colab import files

files.download("/content/submission_AB_cal.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

- 리더보드 점수: 0.33938

- 359등